# Phase 1 — Exploitation du dataset PadChest224 (160K+ images):

## 1.CSV labels:


### Groupe 1 — Identifiants (clés de jointure image ↔ labels):

In [ ]:
import pandas as pd
import ast

df = pd.read_csv(r"C:\Users\gaida\OneDrive - SUPCOM\Desktop\p2m\finale\archive\PADCHEST_chest_x_ray_images_labels_160K_01.02.19.csv", low_memory=False)

# Colonnes clés d'identité
print(df[['ImageID', 'PatientID', 'StudyID', 'StudyDate_DICOM']].head())

# Vérifier le nombre de patients uniques vs images
print(f"Images totales : {len(df)}")
print(f"Patients uniques : {df['PatientID'].nunique()}")
print(f"Études uniques : {df['StudyID'].nunique()}")

### Groupe 2 — Labels:

In [ ]:
# Parser les labels (stockés comme strings de listes Python)
df['Labels'] = df['Labels'].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else [])
df['Localizations'] = df['Localizations'].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else [])

# Voir la distribution des pathologies
from collections import Counter
all_labels = [label for sublist in df['Labels'] for label in sublist]
label_counts = Counter(all_labels)
print(label_counts.most_common(20))

# ⚠️ IMPORTANT : supprimer les labels "Unchanged"
df['Labels'] = df['Labels'].apply(lambda lst: [l for l in lst if l != 'unchanged'])

# Images normales (label 'normal')
normal_mask = df['Labels'].apply(lambda lst: 'normal' in lst)
print(f"Images normales : {normal_mask.sum()}")
print(f"Images pathologiques : {(~normal_mask).sum()}")

### Groupe 3 — Projection & acquisition:

In [ ]:
# Distribution des projections
print(df['Projection'].value_counts())
# PA (Postero-Anterior), AP (Antero-Posterior), L (Lateral), COSTAL, UNK

# ✅ Filtrer uniquement vues frontales (recommandé pour ton stage)
frontal_views = ['PA', 'AP']
df_frontal = df[df['Projection'].isin(frontal_views)].copy()
print(f"Images frontales retenues : {len(df_frontal)}")

# Exclure pédiatrique
df_frontal = df_frontal[df_frontal['Pediatric'] == 'No']

### Groupe 4 — Métadonnées patient (pour analyses EDA):

In [ ]:
# Calcul de l'âge au moment de la radio
df_frontal['StudyDate_DICOM'] = pd.to_datetime(df_frontal['StudyDate_DICOM'], format='%Y%m%d')
df_frontal['PatientBirth'] = pd.to_numeric(df_frontal['PatientBirth'], errors='coerce')
df_frontal['Age'] = df_frontal['StudyDate_DICOM'].dt.year - df_frontal['PatientBirth']

# Distribution par sexe
print(df_frontal['PatientSex_DICOM'].value_counts())

# Exclure adultes seulement (> 18 ans, données fiables)
df_frontal = df_frontal[(df_frontal['Age'] >= 18) & (df_frontal['Age'] <= 100)]

### Binarisation multi-label (transformer les listes de labels en matrice binaire exploitable par un CNN):

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

# Choisir les pathologies cibles (les plus fréquentes et cliniquement pertinentes)
TARGET_LABELS = [
    'normal',
    'pneumonia',
    'nodule',
    'pulmonary fibrosis',
    'pleural effusion',
    'cardiomegaly',
    'consolidation',
    'atelectasis',
    'ground glass pattern',
    'emphysema',
    'pneumothorax',
    'interstitial pattern',
    'bronchiectasis',
    'mass',
    'aortic elongation',
    'pleural thickening',
    'calcified granuloma',
    'mediastinal enlargement',
    'fracture'
]

# Filtrer les labels aux cibles uniquement
df_frontal['Labels_filtered'] = df_frontal['Labels'].apply(
    lambda lst: [l for l in lst if l in TARGET_LABELS]
)

# Supprimer les lignes sans aucun label cible
df_frontal = df_frontal[df_frontal['Labels_filtered'].apply(len) > 0]

# Binarisation → matrice (n_images, n_classes)
mlb = MultiLabelBinarizer(classes=TARGET_LABELS)
y = mlb.fit_transform(df_frontal['Labels_filtered'])

print(f"Shape matrice labels : {y.shape}")
print("Distribution par classe :")
for i, label in enumerate(TARGET_LABELS):
    print(f"  {label}: {y[:, i].sum()} images ({y[:, i].mean()*100:.1f}%)")

### Split patient-level (éviter le data leakage):

In [ ]:
from sklearn.model_selection import train_test_split

# Liste des patients uniques
patients = df_frontal['PatientID'].unique()

# Split patients : 70% train, 15% val, 15% test
train_patients, temp_patients = train_test_split(patients, test_size=0.30, random_state=42)
val_patients, test_patients = train_test_split(temp_patients, test_size=0.50, random_state=42)

# Créer les sous-ensembles
df_train = df_frontal[df_frontal['PatientID'].isin(train_patients)]
df_val   = df_frontal[df_frontal['PatientID'].isin(val_patients)]
df_test  = df_frontal[df_frontal['PatientID'].isin(test_patients)]

print(f"Train : {len(df_train)} images ({df_train['PatientID'].nunique()} patients)")
print(f"Val   : {len(df_val)} images ({df_val['PatientID'].nunique()} patients)")
print(f"Test  : {len(df_test)} images ({df_test['PatientID'].nunique()} patients)")

# ✅ Vérifier l'absence de chevauchement
assert len(set(train_patients) & set(val_patients)) == 0
assert len(set(train_patients) & set(test_patients)) == 0
print("✅ Aucun data leakage patient détecté")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(r"C:\Users\gaida\OneDrive - SUPCOM\Desktop\p2m\finale\archive\PADCHEST_chest_x_ray_images_labels_160K_01.02.19.csv", low_memory=False)

# ── 1. Distribution des projections ──────────────────────────────────────
print("Distribution des projections :")
print(df['Projection'].value_counts())
print(f"\nTotal images : {len(df)}")

# ── 2. Filtrage recommandé pour ton projet ───────────────────────────────
# Garder uniquement PA et AP (vues frontales)
FRONTAL_PROJECTIONS = ['PA', 'AP']
df_frontal = df[df['Projection'].isin(FRONTAL_PROJECTIONS)].copy()

print(f"\nAprès filtrage frontal : {len(df_frontal)} images")
print(f"  PA : {(df_frontal['Projection'] == 'PA').sum()}")
print(f"  AP : {(df_frontal['Projection'] == 'AP').sum()}")

# ── 3. Vérifier si la projection influence la distribution des labels ─────
import ast

df_frontal['Labels'] = df_frontal['Labels'].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else []
)

# Taux de normalité par projection
df_frontal['is_normal'] = df_frontal['Labels'].apply(
    lambda lst: 'normal' in lst
)

print("\nTaux de normalité par projection :")
print(df_frontal.groupby('Projection')['is_normal'].mean().round(3))

# ── 4. Ajouter la projection comme feature dans le DataLoader ─────────────
# Encoder la projection en one-hot pour l'utiliser comme métadonnée
projection_dummies = pd.get_dummies(df_frontal['Projection'], prefix='proj')
df_frontal = pd.concat([df_frontal, projection_dummies], axis=1)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
from collections import Counter

# ── Chargement ─────────────────────────────────────────────────────────────
df = pd.read_csv("/kaggle/input/datasets/wassimfouratelamri/padchest224-complete/PADCHEST_chest_x_ray_images_labels_160K_01.02.19.csv", low_memory=False)

# ── 1. Calcul de l'âge ─────────────────────────────────────────────────────
df['StudyDate_DICOM'] = pd.to_datetime(
    df['StudyDate_DICOM'].astype(str), format='%Y%m%d', errors='coerce'
)
df['PatientBirth'] = pd.to_numeric(df['PatientBirth'], errors='coerce')
df['Age'] = df['StudyDate_DICOM'].dt.year - df['PatientBirth']

# Nettoyage : garder âges plausibles
df = df[(df['Age'] >= 18) & (df['Age'] <= 100) | df['Age'].isna()]

# ── 2. Statistiques démographiques globales ────────────────────────────────
print("=== DÉMOGRAPHIE GLOBALE ===")
print(f"Total images           : {len(df):,}")
print(f"Patients uniques       : {df['PatientID'].nunique():,}")
print(f"Études uniques         : {df['StudyID'].nunique():,}")
print(f"\nSexe :")
print(df['PatientSex_DICOM'].value_counts())
print(f"\nPédiatrique :")
print(df['Pediatric'].value_counts())
print(f"\nÂge — moyenne : {df['Age'].mean():.1f} ± {df['Age'].std():.1f} ans")
print(f"Âge — min/max : {df['Age'].min():.0f} / {df['Age'].max():.0f} ans")

# ── 3. Filtrage standard (adultes frontaux, non-pédiatriques) ──────────────
df_clean = df[
    (df['Projection'].isin(['PA', 'AP'])) &
    (df['Pediatric'] == 'No') &
    (df['Age'] >= 18) &
    (df['Age'] <= 100)
].copy()
print(f"\nAprès filtrage : {len(df_clean):,} images retenues")

# ── 4. Analyse biais démographique par pathologie ─────────────────────────
df_clean['Labels'] = df_clean['Labels'].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else []
)

TARGET_LABELS = [
    'pneumonia', 'nodule', 'pleural effusion', 'cardiomegaly',
    'pneumothorax', 'pulmonary fibrosis', 'consolidation', 'atelectasis'
]

# Créer colonnes binaires par pathologie
for label in TARGET_LABELS:
    df_clean[f'has_{label.replace(" ", "_")}'] = df_clean['Labels'].apply(
        lambda lst: int(label in lst)
    )

# Prévalence par sexe
print("\n=== PRÉVALENCE PAR SEXE (%) ===")
sex_prev = df_clean.groupby('PatientSex_DICOM')[
    [f'has_{l.replace(" ", "_")}' for l in TARGET_LABELS]
].mean() * 100
sex_prev.columns = TARGET_LABELS
print(sex_prev.round(2))

# ── 5. Groupes d'âge et prévalence ────────────────────────────────────────
df_clean['AgeGroup'] = pd.cut(
    df_clean['Age'],
    bins=[18, 40, 60, 80, 100],
    labels=['18-40', '41-60', '61-80', '81-100']
)

print("\n=== PRÉVALENCE PAR GROUPE D'ÂGE (%) ===")
age_prev = df_clean.groupby('AgeGroup', observed=True)[
    [f'has_{l.replace(" ", "_")}' for l in TARGET_LABELS]
].mean() * 100
age_prev.columns = TARGET_LABELS
print(age_prev.round(2))

# ── 6. Études multiples par patient (aspect longitudinal) ─────────────────
studies_per_patient = df_clean.groupby('PatientID')['StudyID'].nunique()
print(f"\n=== ÉTUDES PAR PATIENT ===")
print(f"Médiane : {studies_per_patient.median():.0f} étude(s)")
print(f"Max     : {studies_per_patient.max():.0f} études")
print(f"Patients avec > 3 études : {(studies_per_patient > 3).sum():,}")

# ── 7. Feature engineering : encoder les métadonnées pour le modèle ────────
# Ces features seront concaténées au vecteur CNN avant la couche de classification

def build_metadata_features(df_input):
    """Construit un vecteur de métadonnées normalisé pour chaque image."""
    meta = pd.DataFrame()

    # Âge normalisé [0, 1]
    meta['age_norm'] = (df_input['Age'] - 18) / (100 - 18)

    # Sexe one-hot
    meta['sex_M'] = (df_input['PatientSex_DICOM'] == 'M').astype(float)
    meta['sex_F'] = (df_input['PatientSex_DICOM'] == 'F').astype(float)

    # Projection one-hot
    meta['proj_PA'] = (df_input['Projection'] == 'PA').astype(float)
    meta['proj_AP'] = (df_input['Projection'] == 'AP').astype(float)

    # Année de la radio (tendance temporelle)
    meta['year_norm'] = (df_input['StudyDate_DICOM'].dt.year - 2009) / (2017 - 2009)

    return meta.fillna(0.5)  # Imputation pour valeurs manquantes

meta_features = build_metadata_features(df_clean)
print(f"\n=== FEATURES MÉTADONNÉES ===")
print(meta_features.describe().round(3))

In [ ]:
import torch
import torch.nn as nn
import timm

N_CLASSES = 19
N_META = 5

class PadChestWithMetadata(nn.Module):
    """
    DenseNet121 + métadonnées patient fusionnées avant la classification.
    Architecture : CNN backbone → features image (1024)
                   Metadata MLP → features meta (32)
                   Concat → classifieur final (n_classes)
    """
    def __init__(self, n_classes=N_CLASSES, n_meta_features=6):
        super().__init__()

        # Backbone CNN (DenseNet121 pré-entraîné ImageNet)
        self.backbone = timm.create_model(
            'densenet121', pretrained=True, num_classes=0  # num_classes=0 → pas de tête
        )
        cnn_out_dim = self.backbone.num_features  # 1024 pour DenseNet121

        # Branche métadonnées (MLP simple)
        self.meta_mlp = nn.Sequential(
            nn.Linear(n_meta_features, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU()
        )

        # Classifieur fusion
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(cnn_out_dim + 32, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, n_classes)
        )

    def forward(self, image, metadata):
        # Features visuelles
        img_features = self.backbone(image)           # (B, 1024)

        # Features métadonnées
        meta_features = self.meta_mlp(metadata)       # (B, 32)

        # Fusion par concaténation
        fused = torch.cat([img_features, meta_features], dim=1)  # (B, 1056)

        return self.classifier(fused)                 # (B, n_classes)


# Utilisation dans la boucle d'entraînement
model = PadChestWithMetadata(n_classes=N_CLASSES, n_meta_features=6)

# Exemple de forward pass
batch_images = torch.randn(8, 3, 224, 224)   # batch de 8 images
batch_meta   = torch.randn(8, 6)             # [age_norm, sex_M, sex_F, proj_PA, proj_AP, year_norm]
logits = model(batch_images, batch_meta)
print(f"Output shape : {logits.shape}")      # torch.Size([8, 14])

# Phase 2 — Traitement du dataset (CSV):

## 1. Nettoyage labels (Suppr. Unchanged, NaN, Multi-label binarisation):

In [ ]:
import pandas as pd
import ast
import numpy as np

# ── Chargement ────────────────────────────────────────────────────────────────
df = pd.read_csv(
    "/kaggle/input/datasets/wassimfouratelamri/padchest224-complete/PADCHEST_chest_x_ray_images_labels_160K_01.02.19.csv",
    low_memory=False
)

# Parser les labels (string → liste Python)
df['Labels'] = df['Labels'].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else []
)

# Date pour trier les études chronologiquement
df['StudyDate_DICOM'] = pd.to_datetime(
    df['StudyDate_DICOM'].astype(str), format='%Y%m%d', errors='coerce'
)

print(f"Images initiales : {len(df):,}")
print(f"Images avec 'unchanged' : {df['Labels'].apply(lambda l: 'unchanged' in l).sum():,}")

# ── Étape 1 : trier par patient et par date ───────────────────────────────────
df = df.sort_values(['PatientID', 'StudyDate_DICOM']).reset_index(drop=True)

# ── Étape 2 : résoudre les Unchanged par propagation arrière ─────────────────
def resolve_unchanged(df):
    """
    Pour chaque image ayant le label 'unchanged' :
    - Cherche l'étude précédente du même patient (date antérieure)
    - Si trouvée et sans 'unchanged' → remplace les labels
    - Sinon → marque pour suppression (labels vides)
    """
    df = df.copy()
    df['Labels_resolved'] = df['Labels'].copy()
    df['was_unchanged'] = False
    df['unchanged_resolved'] = False

    # Grouper par patient
    for patient_id, group in df.groupby('PatientID'):
        group_sorted = group.sort_values('StudyDate_DICOM')
        indices = group_sorted.index.tolist()

        for i, idx in enumerate(indices):
            labels = df.at[idx, 'Labels']

            if 'unchanged' not in labels:
                continue

            df.at[idx, 'was_unchanged'] = True

            # Chercher une étude précédente valide (sans unchanged)
            prior_labels = None
            for j in range(i - 1, -1, -1):
                prior_idx = indices[j]
                prior = df.at[prior_idx, 'Labels_resolved']
                if 'unchanged' not in prior and len(prior) > 0:
                    prior_labels = prior
                    break

            if prior_labels is not None:
                # ✅ Remplacer par les labels de l'étude antérieure
                # Fusionner : labels actuels (sans 'unchanged') + labels antérieurs
                current_clean = [l for l in labels if l != 'unchanged']
                merged = list(set(current_clean + prior_labels))
                df.at[idx, 'Labels_resolved'] = merged
                df.at[idx, 'unchanged_resolved'] = True
            else:
                # ❌ Pas d'étude antérieure → supprimer (labels vides)
                df.at[idx, 'Labels_resolved'] = []

    return df

df = resolve_unchanged(df)

# ── Étape 3 : statistiques de résolution ─────────────────────────────────────
n_unchanged     = df['was_unchanged'].sum()
n_resolved      = df['unchanged_resolved'].sum()
n_dropped       = n_unchanged - n_resolved
n_empty         = df['Labels_resolved'].apply(len) == 0

print(f"\n=== RÉSOLUTION UNCHANGED ===")
print(f"Images avec 'unchanged'     : {n_unchanged:,}")
print(f"  → Résolues (étude antér.) : {n_resolved:,} ({100*n_resolved/n_unchanged:.1f}%)")
print(f"  → Supprimées (aucune réf) : {n_dropped:,} ({100*n_dropped/n_unchanged:.1f}%)")
print(f"Images sans aucun label     : {n_empty.sum():,}")
# Vérifier d'où viennent les 156 restants
mask_remaining = df_clean['Labels'].apply(lambda l: 'unchanged' in l)
print(f"Images avec 'unchanged' résiduel : {mask_remaining.sum()}")

# Exemple : voir leurs labels
print("\nExemples de labels résiduels :")
print(df_clean[mask_remaining]['Labels'].head(10).tolist())

# ── Fix : retirer 'unchanged' de force sur toutes les images ─────────────────
df_clean['Labels'] = df_clean['Labels'].apply(
    lambda lst: [l for l in lst if l != 'unchanged']
)

# Supprimer les images qui deviennent vides après ce retrait
n_before = len(df_clean)
df_clean = df_clean[df_clean['Labels'].apply(len) > 0].copy()
print(f"\nImages supprimées après fix résiduel : {n_before - len(df_clean)}")

# ── Vérification finale ───────────────────────────────────────────────────────
n_remaining = df_clean['Labels'].apply(lambda l: 'unchanged' in l).sum()
assert n_remaining == 0, f"ERREUR : {n_remaining} restants"
print(f"✅ Aucun label 'unchanged' restant")
print(f"✅ Dataset propre : {len(df_clean):,} images")


# ── Étape 4 : supprimer les images sans labels valides ───────────────────────
df_clean = df[df['Labels_resolved'].apply(len) > 0].copy()
df_clean['Labels'] = df_clean['Labels_resolved']
df_clean = df_clean.drop(columns=['Labels_resolved', 'was_unchanged', 'unchanged_resolved'])

print(f"\nImages après nettoyage : {len(df_clean):,}")
print(f"Images supprimées      : {len(df) - len(df_clean):,}")

# ── Étape 5 : autres nettoyages complémentaires ───────────────────────────────

# 5a. Supprimer le label 'exclude' (images de mauvaise qualité)
df_clean['Labels'] = df_clean['Labels'].apply(
    lambda lst: [l for l in lst if l != 'exclude']
)
n_after_exclude = (df_clean['Labels'].apply(len) == 0).sum()
df_clean = df_clean[df_clean['Labels'].apply(len) > 0].copy()
print(f"Supprimées après retrait 'exclude' : {n_after_exclude:,}")

# 5b. Normaliser les labels (minuscules, strip espaces)
df_clean['Labels'] = df_clean['Labels'].apply(
    lambda lst: [l.strip().lower() for l in lst]
)

# 5c. Dédupliquer les labels par image
df_clean['Labels'] = df_clean['Labels'].apply(
    lambda lst: list(set(lst))
)

# ── Étape 6 : vérification finale ────────────────────────────────────────────
# Vérifier qu'aucun 'unchanged' ne subsiste
n_remaining = df_clean['Labels'].apply(lambda l: 'unchanged' in l).sum()

print(f"\n✅ Aucun label 'unchanged' restant")
print(f"✅ Dataset propre : {len(df_clean):,} images prêtes")

# ── Étape 7 : sauvegarder ────────────────────────────────────────────────────
# Reconvertir les listes en strings pour sauvegarde CSV
df_clean['Labels'] = df_clean['Labels'].apply(str)
df_clean.to_csv("/kaggle/working/padchest_labels_clean.csv", index=False)
print("✅ Sauvegardé → padchest_labels_clean.csv")

## 2. Équilibrage classes (Class weights, oversampling, Focal loss pour minorités):

### Étape 1 — Analyser le déséquilibre:

In [ ]:
import pandas as pd
import numpy as np
import ast
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.preprocessing import MultiLabelBinarizer

df = pd.read_csv("/kaggle/working/padchest_labels_clean.csv")
df['Labels'] = df['Labels'].apply(ast.literal_eval)

TARGET_LABELS = [
    'normal',
    'pneumonia',
    'nodule',
    'pulmonary fibrosis',
    'pleural effusion',
    'cardiomegaly',
    'consolidation',
    'atelectasis',
    'ground glass pattern',
    'emphysema',
    'pneumothorax',
    'interstitial pattern',
    'bronchiectasis',
    'mass',
    'aortic elongation',
    'pleural thickening',
    'calcified granuloma',
    'mediastinal enlargement',
    'fracture'
]

# Filtrer labels aux cibles
df['Labels'] = df['Labels'].apply(
    lambda lst: [l for l in lst if l in TARGET_LABELS]
)
df = df[df['Labels'].apply(len) > 0].copy()

# Binarisation
mlb = MultiLabelBinarizer(classes=TARGET_LABELS)
y = mlb.fit_transform(df['Labels'])

# Statistiques déséquilibre
counts = y.sum(axis=0)
freqs  = counts / len(y)
imbalance_ratio = counts.max() / counts  # ratio max_class / chaque classe

print(f"{'Label':<30} {'Count':>8} {'Freq%':>8} {'Imbalance':>10}")
print("-" * 60)
for i, label in enumerate(TARGET_LABELS):
    print(f"{label:<30} {int(counts[i]):>8,} {freqs[i]*100:>7.2f}% {imbalance_ratio[i]:>10.1f}x")

### Étape 2 — Trois stratégies combinées:

#### Stratégie A : Class weights (obligatoire, coût zéro):

In [ ]:
import torch

def compute_class_weights(y, method='effective'):
    """
    Calcule les poids par classe pour BCEWithLogitsLoss.
    method='effective' : Cui et al. 2019 — meilleur que l'inverse simple
    method='inverse'   : 1 / freq (baseline classique)
    """
    counts = y.sum(axis=0)
    n_samples = len(y)

    if method == 'inverse':
        weights = n_samples / (len(TARGET_LABELS) * counts)

    elif method == 'effective':
        # Effective Number of Samples (Cui et al. CVPR 2019)
        beta = 0.9999
        effective_num = 1.0 - np.power(beta, counts)
        weights = (1.0 - beta) / effective_num
        # Normaliser pour que la moyenne = 1
        weights = weights / weights.mean()

    return torch.tensor(weights, dtype=torch.float32)

pos_weights = compute_class_weights(y, method='effective')

print("Class weights calculés :")
for i, label in enumerate(TARGET_LABELS):
    print(f"  {label:<30} weight = {pos_weights[i].item():.3f}")

# Utilisation dans la loss
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weights)
# pos_weight amplifie la pénalité sur les faux négatifs des classes rares

#### Stratégie B : Focal Loss (remplace BCEWithLogitsLoss):

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    """
    Focal Loss multi-label pour déséquilibre sévère.
    Lin et al. 2017 — adapté au multi-label (sigmoid, pas softmax).

    gamma=2  : focalisation standard (recommandé)
    alpha    : class weights optionnels (tensor de taille n_classes)
    """
    def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
        super().__init__()
        self.gamma     = gamma
        self.alpha     = alpha  # tensor (n_classes,) ou None
        self.reduction = reduction

    def forward(self, logits, targets):
        # logits  : (B, C) — raw scores avant sigmoid
        # targets : (B, C) — binaire 0/1
        probs   = torch.sigmoid(logits)
        bce     = F.binary_cross_entropy_with_logits(
            logits, targets, reduction='none'
        )  # (B, C)

        # Facteur focal : (1 - p_t)^gamma
        p_t     = probs * targets + (1 - probs) * (1 - targets)
        focal   = (1 - p_t) ** self.gamma

        loss = focal * bce  # (B, C)

        # Appliquer alpha (class weights) si fourni
        if self.alpha is not None:
            alpha = self.alpha.to(logits.device)
            loss = loss * alpha.unsqueeze(0)  # broadcast sur le batch

        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss


# ── Utilisation ───────────────────────────────────────────────────────────────
# Option 1 : Focal Loss seule
criterion = FocalLoss(gamma=2.0)

# Option 2 : Focal Loss + class weights (recommandé pour PadChest)
criterion = FocalLoss(gamma=2.0, alpha=pos_weights)

# Test
logits  = torch.randn(8, 14)
targets = torch.randint(0, 2, (8, 14)).float()
loss    = criterion(logits, targets)
print(f"Focal Loss test : {loss.item():.4f}")

#### Stratégie C : Oversampling des classes rares (WeightedRandomSampler):

In [ ]:
from torch.utils.data import WeightedRandomSampler

def compute_sample_weights(y, target_labels):
    """
    Attribue à chaque image un poids de tirage proportionnel
    à la rareté de ses pathologies les plus rares.
    """
    counts   = y.sum(axis=0)                      # (n_classes,)
    freqs    = counts / len(y)
    # Poids de classe = inverse de la fréquence
    class_w  = 1.0 / (freqs + 1e-6)

    # Poids de chaque image = max des poids de ses classes positives
    # (on privilégie les images avec des pathologies rares)
    sample_w = np.zeros(len(y))
    for i in range(len(y)):
        pos_classes = np.where(y[i] == 1)[0]
        if len(pos_classes) > 0:
            sample_w[i] = class_w[pos_classes].max()
        else:
            sample_w[i] = 1.0  # images sans label cible → poids neutre

    return sample_w

sample_weights = compute_sample_weights(y, TARGET_LABELS)

# Intégration dans DataLoader
sampler = WeightedRandomSampler(
    weights     = torch.DoubleTensor(sample_weights),
    num_samples = len(sample_weights),
    replacement = True   # tirage avec remise pour les classes rares
)

# DataLoader avec sampler (remplace shuffle=True)
from torch.utils.data import DataLoader
# loader = DataLoader(dataset, batch_size=32, sampler=sampler, num_workers=4)
# ⚠️ Ne pas mettre shuffle=True si sampler est défini

print(f"Sample weights — min : {sample_weights.min():.2f}")
print(f"Sample weights — max : {sample_weights.max():.2f}")
print(f"Sample weights — moy : {sample_weights.mean():.2f}")

### Étape 3 — Label Smoothing (régularisation):

In [ ]:
class BCELossWithLabelSmoothing(nn.Module):
    """
    BCE + label smoothing pour multi-label.
    smoothing=0.1 : les 1 deviennent 0.9, les 0 deviennent 0.1
    Réduit l'overconfidence sur les classes majoritaires.
    """
    def __init__(self, smoothing=0.1, pos_weight=None):
        super().__init__()
        self.smoothing  = smoothing
        self.pos_weight = pos_weight

    def forward(self, logits, targets):
        # Smooth les targets
        targets_smooth = targets * (1 - self.smoothing) + 0.5 * self.smoothing
        return F.binary_cross_entropy_with_logits(
            logits, targets_smooth,
            pos_weight=self.pos_weight,
            reduction='mean'
        )

criterion_smooth = BCELossWithLabelSmoothing(smoothing=0.1, pos_weight=pos_weights)

### Étape 4 — Seuils optimaux par classe (post-entraînement):

In [ ]:
from sklearn.metrics import f1_score, roc_curve
import numpy as np

def find_optimal_thresholds(y_true, y_probs, target_labels, metric='f1'):
    """
    Cherche le seuil optimal par classe en maximisant F1
    sur le set de validation.
    """
    thresholds = {}
    candidate_thresholds = np.arange(0.1, 0.9, 0.02)

    print(f"{'Label':<30} {'Seuil opt.':>10} {'F1 @ seuil':>12} {'F1 @ 0.5':>10}")
    print("-" * 65)

    for i, label in enumerate(target_labels):
        best_thresh = 0.5
        best_f1     = 0.0

        for thresh in candidate_thresholds:
            preds = (y_probs[:, i] >= thresh).astype(int)
            f1    = f1_score(y_true[:, i], preds, zero_division=0)
            if f1 > best_f1:
                best_f1    = f1
                best_thresh = thresh

        # F1 au seuil 0.5 pour comparaison
        preds_05 = (y_probs[:, i] >= 0.5).astype(int)
        f1_05    = f1_score(y_true[:, i], preds_05, zero_division=0)

        thresholds[label] = best_thresh
        print(f"{label:<30} {best_thresh:>10.2f} {best_f1:>12.4f} {f1_05:>10.4f}")

    return thresholds


# Après entraînement, sur le set de validation :
# thresholds = find_optimal_thresholds(y_val, y_probs_val, TARGET_LABELS)

# Prédictions finales avec seuils optimaux
def predict_with_thresholds(y_probs, thresholds, target_labels):
    preds = np.zeros_like(y_probs)
    for i, label in enumerate(target_labels):
        preds[:, i] = (y_probs[:, i] >= thresholds[label]).astype(int)
    return preds

 ## 3. Split patient-level (Train 70 / Val 15 / Test 15) Pas de data leakage patient:

In [ ]:
import pandas as pd
import numpy as np
import ast
from collections import Counter
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
%pip install iterative-stratification


from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
# pip install iterative-stratification

df = pd.read_csv("/kaggle/working/padchest_labels_clean.csv")
df['Labels'] = df['Labels'].apply(ast.literal_eval)

TARGET_LABELS = [
    'normal',
    'pneumonia',
    'nodule',
    'pulmonary fibrosis',
    'pleural effusion',
    'cardiomegaly',
    'consolidation',
    'atelectasis',
    'ground glass pattern',
    'emphysema',
    'pneumothorax',
    'interstitial pattern',
    'bronchiectasis',
    'mass',
    'aortic elongation',
    'pleural thickening',
    'calcified granuloma',
    'mediastinal enlargement',
    'fracture'
]

df['Labels'] = df['Labels'].apply(
    lambda lst: [l for l in lst if l in TARGET_LABELS]
)
df = df[df['Labels'].apply(len) > 0].copy().reset_index(drop=True)

mlb = MultiLabelBinarizer(classes=TARGET_LABELS)
y_full = mlb.transform(df['Labels'])

print(f"Images totales : {len(df):,}")
print(f"Patients uniques : {df['PatientID'].nunique():,}")

In [ ]:
# Un patient = une ligne avec TOUS ses labels agrégés
patient_df = (
    df.groupby('PatientID')['Labels']
    .apply(lambda series: list(set(
        label for lst in series for label in lst
    )))
    .reset_index()
)
patient_df.columns = ['PatientID', 'AllLabels']

# Binarisation au niveau patient (pour stratification)
y_patient = mlb.transform(patient_df['AllLabels'])

print(f"Patients à splitter : {len(patient_df):,}")
print(f"Shape matrice patient : {y_patient.shape}")

In [ ]:
# ── Split 1 : train (70%) vs temp (30%) ──────────────────────────────────────
msss = MultilabelStratifiedShuffleSplit(
    n_splits=1, test_size=0.30, random_state=42
)
train_idx, temp_idx = next(msss.split(
    np.zeros(len(patient_df)), y_patient
))

train_patients = patient_df.iloc[train_idx]['PatientID'].values
temp_patients  = patient_df.iloc[temp_idx]['PatientID'].values

# ── Split 2 : val (15%) vs test (15%) sur temp ───────────────────────────────
y_temp = y_patient[temp_idx]
patient_temp_df = patient_df.iloc[temp_idx].reset_index(drop=True)

msss2 = MultilabelStratifiedShuffleSplit(
    n_splits=1, test_size=0.50, random_state=42
)
val_idx_rel, test_idx_rel = next(msss2.split(
    np.zeros(len(patient_temp_df)), y_temp
))

val_patients  = patient_temp_df.iloc[val_idx_rel]['PatientID'].values
test_patients = patient_temp_df.iloc[test_idx_rel]['PatientID'].values

print(f"\n=== SPLIT PATIENTS ===")
print(f"Train : {len(train_patients):,} patients")
print(f"Val   : {len(val_patients):,} patients")
print(f"Test  : {len(test_patients):,} patients")

In [ ]:
# Mapper les patients → images
df_train = df[df['PatientID'].isin(train_patients)].copy()
df_val   = df[df['PatientID'].isin(val_patients)].copy()
df_test  = df[df['PatientID'].isin(test_patients)].copy()

print(f"\n=== SPLIT IMAGES ===")
print(f"Train : {len(df_train):,} images")
print(f"Val   : {len(df_val):,} images")
print(f"Test  : {len(df_test):,} images")
print(f"Total : {len(df_train)+len(df_val)+len(df_test):,} / {len(df):,}")

In [ ]:
def verify_no_leakage(df_train, df_val, df_test):
    train_p = set(df_train['PatientID'])
    val_p   = set(df_val['PatientID'])
    test_p  = set(df_test['PatientID'])

    leak_tv = train_p & val_p
    leak_tt = train_p & test_p
    leak_vt = val_p   & test_p

    print("\n=== VÉRIFICATION DATA LEAKAGE ===")
    print(f"Patients communs Train∩Val  : {len(leak_tv)}")
    print(f"Patients communs Train∩Test : {len(leak_tt)}")
    print(f"Patients communs Val∩Test   : {len(leak_vt)}")

    assert len(leak_tv) == 0, f"LEAKAGE Train/Val : {len(leak_tv)} patients"
    assert len(leak_tt) == 0, f"LEAKAGE Train/Test : {len(leak_tt)} patients"
    assert len(leak_vt) == 0, f"LEAKAGE Val/Test : {len(leak_vt)} patients"

    # Vérifier couverture complète
    all_split = train_p | val_p | test_p
    all_orig  = set(df['PatientID'])
    missing   = all_orig - all_split
    print(f"Patients non assignés       : {len(missing)}")
    print("✅ Aucun data leakage détecté")

verify_no_leakage(df_train, df_val, df_test)

In [ ]:
def check_stratification(df_train, df_val, df_test, target_labels, mlb):
    y_tr = mlb.transform(df_train['Labels'])
    y_va = mlb.transform(df_val['Labels'])
    y_te = mlb.transform(df_test['Labels'])

    n_tr, n_va, n_te = len(y_tr), len(y_va), len(y_te)

    print(f"\n=== DISTRIBUTION PAR PATHOLOGIE ===")
    print(f"{'Label':<30} {'Train%':>8} {'Val%':>8} {'Test%':>8} {'Drift':>8}")
    print("-" * 66)

    drifts = []
    for i, label in enumerate(target_labels):
        p_tr = y_tr[:, i].sum() / n_tr * 100
        p_va = y_va[:, i].sum() / n_va * 100
        p_te = y_te[:, i].sum() / n_te * 100
        # Drift = écart max entre les 3 sets
        drift = max(abs(p_tr - p_va), abs(p_tr - p_te), abs(p_va - p_te))
        drifts.append(drift)
        flag = "⚠️" if drift > 1.5 else "✅"
        print(f"{label:<30} {p_tr:>7.2f}% {p_va:>7.2f}% {p_te:>7.2f}% "
              f"{drift:>6.2f}pp {flag}")

    print(f"\nDrift moyen  : {np.mean(drifts):.3f} pp")
    print(f"Drift max    : {np.max(drifts):.3f} pp (sur '{target_labels[np.argmax(drifts)]}')")
    print("Objectif     : drift < 1.5 pp par classe")

check_stratification(df_train, df_val, df_test, TARGET_LABELS, mlb)

In [ ]:
# Ajouter colonne split et sauvegarder en un seul fichier
df_train['split'] = 'train'
df_val['split']   = 'val'
df_test['split']  = 'test'

df_final = pd.concat([df_train, df_val, df_test], ignore_index=True)
df_final['Labels'] = df_final['Labels'].apply(str)
df_final.to_csv("/kaggle/working/padchest_split.csv", index=False)

print(f"\n✅ Sauvegardé → padchest_split.csv")
print(f"   Train : {(df_final['split']=='train').sum():,} images")
print(f"   Val   : {(df_final['split']=='val').sum():,} images")
print(f"   Test  : {(df_final['split']=='test').sum():,} images")

# Recharger proprement
df_split = pd.read_csv("/kaggle/working/padchest_split.csv")
df_split['Labels'] = df_split['Labels'].apply(ast.literal_eval)

df_train_final = df_split[df_split['split'] == 'train']
df_val_final   = df_split[df_split['split'] == 'val']
df_test_final  = df_split[df_split['split'] == 'test']
print("✅ Rechargement OK — prêt pour le DataLoader")

# Phase 3 — Traitement des images (preprocessing):

## 1. Normalisation Pixel [0,1], mean/std ImageNet stats (gris→RGB) CLAHE si nécessaire:



### Étape 1 — Comprendre le windowing:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2

def apply_window(image_array, window_center, window_width):
    """
    Windowing DICOM : clip les valeurs autour du centre clinique.
    Simule ce que le radiologue voit sur son écran DICOM.

    window_center (WC) : valeur centrale de la fenêtre
    window_width  (WW) : étendue de la fenêtre

    Plage visible : [WC - WW/2 , WC + WW/2]
    Valeurs en dessous → noir (0)
    Valeurs au dessus  → blanc (255)
    """
    low  = window_center - window_width / 2
    high = window_center + window_width / 2

    # Clip
    img_windowed = np.clip(image_array, low, high)

    # Normaliser vers [0, 255]
    img_windowed = (img_windowed - low) / (high - low) * 255.0

    return img_windowed.astype(np.uint8)


# ── Fenêtres standards pour thorax ────────────────────────────────────────────
WINDOWS = {
    'pulmonaire' : {'WC':  -600, 'WW': 1500},  # parenchyme pulmonaire
    'mediastin'  : {'WC':    40, 'WW':  400},  # médiastin, cœur
    'os'         : {'WC':   400, 'WW': 1800},  # côtes, vertèbres
    'defaut'     : {'WC': -  0,  'WW': 2000},  # vue globale chest
}

# Pour PadChest PNG 224 : utiliser les WindowCenter/WindowWidth du CSV
# df['WindowCenter_DICOM'] et df['WindowWidth_DICOM']

### Étape 2 — Pipeline complet de normalisation:

In [ ]:
import torch
import torchvision.transforms as T
from PIL import Image
import numpy as np
import cv2

# ── Constantes ImageNet (gris → RGB repliqué) ─────────────────────────────────
# PadChest est en niveaux de gris → on réplique sur 3 canaux
# Les stats ImageNet restent valables car les backbones pré-entraînés
# s'attendent à ces valeurs
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Stats spécifiques PadChest (calculées sur le train set)
# Plus précis que ImageNet si tu entraînes from scratch
PADCHEST_MEAN = [0.503, 0.503, 0.503]  # ~0.5 car images médicales centrées
PADCHEST_STD  = [0.246, 0.246, 0.246]


def load_and_normalize(image_path, size=224, use_clahe=True,
                       use_imagenet_stats=True):
    """
    Pipeline complet : chargement → CLAHE → RGB → normalisation.
    Retourne un tensor (3, 224, 224) prêt pour le CNN.
    """
    # 1. Charger en niveaux de gris
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Image non trouvée : {image_path}")

    # 2. Resize si nécessaire (déjà 224 dans PadChest224)
    if img.shape[0] != size or img.shape[1] != size:
        img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)

    # 3. CLAHE — améliore le contraste local sans saturer
    if use_clahe:
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        img   = clahe.apply(img)

    # 4. Normaliser pixels → [0.0, 1.0]
    img = img.astype(np.float32) / 255.0

    # 5. Répliquer sur 3 canaux (grayscale → pseudo-RGB)
    img = np.stack([img, img, img], axis=0)  # (3, H, W)

    # 6. Normalisation ImageNet (ou PadChest)
    mean = IMAGENET_MEAN if use_imagenet_stats else PADCHEST_MEAN
    std  = IMAGENET_STD  if use_imagenet_stats else PADCHEST_STD

    for c in range(3):
        img[c] = (img[c] - mean[c]) / std[c]

    return torch.tensor(img, dtype=torch.float32)  # (3, 224, 224)

### Étape 3 — Calculer les stats réelles du train set:

In [ ]:
from torch.utils.data import DataLoader, Dataset
from pathlib import Path
from tqdm import tqdm

class PadChestRaw(Dataset):
    """Dataset minimal pour calculer mean/std sur le train set."""
    def __init__(self, df, image_dir):
        self.paths = [
            Path(image_dir) / row['ImageDir'] / row['ImageID']
            for _, row in df.iterrows()
        ]

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = cv2.imread(str(self.paths[idx]), cv2.IMREAD_GRAYSCALE)
        if img is None:
            return torch.zeros(1, 224, 224)
        img = cv2.resize(img, (224, 224))
        img = img.astype(np.float32) / 255.0
        return torch.tensor(img).unsqueeze(0)  # (1, 224, 224)


def compute_dataset_stats(df_train, image_dir, batch_size=256, num_workers=4):
    """
    Calcule mean et std pixel du train set en une passe.
    À faire une seule fois et sauvegarder les valeurs.
    """
    dataset = PadChestRaw(df_train, image_dir)
    loader  = DataLoader(dataset, batch_size=batch_size,
                         num_workers=num_workers, shuffle=False)

    mean  = 0.0
    var   = 0.0
    count = 0

    print("Calcul des statistiques du train set...")
    for batch in tqdm(loader):
        b = batch.size(0)
        mean  += batch.mean([0, 2, 3]).sum().item() * b
        var   += batch.var([0, 2, 3]).sum().item() * b
        count += b

    mean /= count
    std   = np.sqrt(var / count)

    print(f"\nMean pixel train set : {mean:.4f}")
    print(f"Std  pixel train set : {std:.4f}")
    print("\nUtilise ces valeurs dans ton DataLoader :")
    print(f"  MEAN = [{mean:.4f}, {mean:.4f}, {mean:.4f}]")
    print(f"  STD  = [{std:.4f}, {std:.4f}, {std:.4f}]")

    return mean, std

# IMAGE_DIR = "/kaggle/input/padchest224-complete/images"
# mean, std = compute_dataset_stats(df_train_final, IMAGE_DIR)

### Étape 4 — Transforms train vs val/test:

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

MEAN = [0.503, 0.503, 0.503]
STD  = [0.246, 0.246, 0.246]

# ── Train : augmentations + normalisation ─────────────────────────────────────
train_transform = A.Compose([
    A.Resize(224, 224),

    # Augmentations cliniquement plausibles
    A.HorizontalFlip(p=0.5),              # seule flip valide (pas vertical)
    A.Rotate(limit=10, p=0.5),            # légère rotation (patient pas toujours droit)
    A.RandomBrightnessContrast(           # variation d'exposition radio
        brightness_limit=0.15,
        contrast_limit=0.15,
        p=0.4
    ),
    A.GaussNoise(var_limit=(5, 20), p=0.3),  # bruit détecteur
    A.GridDistortion(num_steps=5,            # légère déformation thorax
                     distort_limit=0.05,
                     p=0.2),
    A.CLAHE(clip_limit=2.0,                  # amélioration contraste local
            tile_grid_size=(8, 8),
            p=0.3),

    # Normalisation finale
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])

# ── Val / Test : uniquement normalisation (pas d'augmentation) ────────────────
val_transform = A.Compose([
    A.Resize(224, 224),
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),  # toujours CLAHE
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])

In [ ]:
from pathlib import Path
import pandas as pd

IMAGE_DIR = Path("/kaggle/input/datasets/wassimfouratelamri/padchest224-complete/images-224/images-224")

# Voir les sous-dossiers disponibles
subdirs = sorted([p.name for p in IMAGE_DIR.iterdir() if p.is_dir()])
print(f"Sous-dossiers trouvés ({len(subdirs)}) : {subdirs[:10]} ...")

# Vérifier ce que contient ImageDir dans le CSV
print(f"\nValeurs ImageDir (dtype={df['ImageDir'].dtype}) :")
print(df['ImageDir'].value_counts().head(10))

# Construire un chemin exemple et vérifier qu'il existe
row = df.iloc[0]
test_path = IMAGE_DIR / str(int(row['ImageDir'])) / str(row['ImageID'])
print(f"\nChemin test : {test_path}")
print(f"Existe      : {test_path.exists()}")

### Étape 5 — Dataset PyTorch complet:

In [ ]:
import pandas as pd
import numpy as np
import ast
import cv2
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ═══════════════════════════════════════════════════════════════════════════════
# 1. CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════
IMAGE_DIR  = Path("/kaggle/input/datasets/wassimfouratelamri/padchest224-complete/images-224/images-224")
CSV_PATH   = "/kaggle/input/datasets/wassimfouratelamri/padchest224-complete/PADCHEST_chest_x_ray_images_labels_160K_01.02.19.csv"
BATCH_SIZE  = 32
NUM_WORKERS = 2

TARGET_LABELS = [
    'normal',
    'pneumonia',
    'nodule',
    'pulmonary fibrosis',
    'pleural effusion',
    'cardiomegaly',
    'consolidation',
    'atelectasis',
    'ground glass pattern',
    'emphysema',
    'pneumothorax',
    'interstitial pattern',
    'bronchiectasis',
    'mass',
    'aortic elongation',
    'pleural thickening',
    'calcified granuloma',
    'mediastinal enlargement',
    'fracture'
]

MEAN = [0.503, 0.503, 0.503]
STD  = [0.246, 0.246, 0.246]

# ═══════════════════════════════════════════════════════════════════════════════
# 2. CHARGEMENT & NETTOYAGE CSV
# ═══════════════════════════════════════════════════════════════════════════════
print("Chargement du CSV...")
df = pd.read_csv(CSV_PATH, low_memory=False)

df['Labels'] = df['Labels'].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else []
)
df['StudyDate_DICOM'] = pd.to_datetime(
    df['StudyDate_DICOM'].astype(str), format='%Y%m%d', errors='coerce'
)
df['PatientBirth'] = pd.to_numeric(df['PatientBirth'], errors='coerce')
df['Age'] = df['StudyDate_DICOM'].dt.year - df['PatientBirth']

df['Labels'] = df['Labels'].apply(
    lambda lst: [l for l in lst if l not in ('unchanged', 'exclude')]
)
df = df[
    (df['Projection'].isin(['PA', 'AP'])) &
    (df['Pediatric'] == 'No') &
    (df['Age'] >= 18) &
    (df['Age'] <= 100)
].copy()
df['Labels'] = df['Labels'].apply(
    lambda lst: [l.strip().lower() for l in lst if l.strip().lower() in TARGET_LABELS]
)
df = df[df['Labels'].apply(len) > 0].copy().reset_index(drop=True)

# ── Vérifier que le chemin image fonctionne ───────────────────────────────────
row0 = df.iloc[0]
test_path = IMAGE_DIR / str(int(row0['ImageDir'])) / str(row0['ImageID'])
print(f"Chemin test : {test_path}")
print(f"Existe      : {test_path.exists()}")
# Si False → on cherche automatiquement le bon chemin
if not test_path.exists():
    found = list(IMAGE_DIR.parent.rglob(str(row0['ImageID'])))
    if found:
        print(f"Image trouvée ici : {found[0]}")
        IMAGE_DIR = found[0].parent.parent  # remonter de 2 niveaux (subdir/image)
        print(f"IMAGE_DIR corrigé : {IMAGE_DIR}")
    else:
        print("⚠️  Image introuvable — vérifie le montage Kaggle")

print(f"\nImages après nettoyage : {len(df):,}")
print(f"Patients uniques       : {df['PatientID'].nunique():,}")

# ═══════════════════════════════════════════════════════════════════════════════
# 3. BINARISATION + SPLIT
# ═══════════════════════════════════════════════════════════════════════════════
mlb = MultiLabelBinarizer(classes=TARGET_LABELS)

patients = df['PatientID'].unique()
train_p, temp_p = train_test_split(patients, test_size=0.30, random_state=42)
val_p,  test_p  = train_test_split(temp_p,   test_size=0.50, random_state=42)

df_train_final = df[df['PatientID'].isin(train_p)].copy()
df_val_final   = df[df['PatientID'].isin(val_p)].copy()
df_test_final  = df[df['PatientID'].isin(test_p)].copy()

assert len(set(train_p) & set(val_p))  == 0
assert len(set(train_p) & set(test_p)) == 0
assert len(set(val_p)   & set(test_p)) == 0
print(f"Train : {len(df_train_final):,} | Val : {len(df_val_final):,} | Test : {len(df_test_final):,}")

# ═══════════════════════════════════════════════════════════════════════════════
# 4. TRANSFORMS
# ═══════════════════════════════════════════════════════════════════════════════
train_transform = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=10, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
    A.GaussNoise(var_limit=(5, 20), p=0.3),
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.5),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])
val_transform = A.Compose([
    A.Resize(224, 224),
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])

# ═══════════════════════════════════════════════════════════════════════════════
# 5. DATASET — __getitem__ avec conversion str() explicite
# ═══════════════════════════════════════════════════════════════════════════════
class PadChestDataset(Dataset):
    def __init__(self, df, image_dir, mlb, transform=None, return_meta=True):
        self.df          = df.reset_index(drop=True)
        self.image_dir   = Path(image_dir)          # ← Path stocké ici
        self.mlb         = mlb
        self.transform   = transform
        self.return_meta = return_meta
        self.y           = mlb.transform(self.df['Labels'].tolist())
        if return_meta:
            self._build_meta()

    def _build_meta(self):
        df   = self.df
        age  = pd.to_numeric(df['Age'],  errors='coerce').fillna(61)
        sex  = df['PatientSex_DICOM'].fillna('M')
        proj = df['Projection'].fillna('PA')
        self.meta = np.column_stack([
            ((age - 18) / 82).clip(0, 1).values,
            (sex  == 'M').astype(float).values,
            (sex  == 'F').astype(float).values,
            (proj == 'PA').astype(float).values,
            (proj == 'AP').astype(float).values,
        ]).astype(np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # ── Conversion explicite en str pour éviter TypeError ─────────────────
        subdir   = str(int(float(str(row['ImageDir']))))  # int → "54"
        filename = str(row['ImageID'])                    # filename.png

        img_path = self.image_dir / subdir / filename
        img      = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)

        if img is None:
            img = np.zeros((224, 224), dtype=np.uint8)

        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

        if self.transform:
            img = self.transform(image=img)['image']

        label = torch.tensor(self.y[idx], dtype=torch.float32)

        if self.return_meta:
            meta = torch.tensor(self.meta[idx], dtype=torch.float32)
            return img, meta, label

        return img, label

# ═══════════════════════════════════════════════════════════════════════════════
# 6. DATALOADERS
# ═══════════════════════════════════════════════════════════════════════════════
train_dataset = PadChestDataset(df_train_final, IMAGE_DIR, mlb,
                                transform=train_transform, return_meta=True)
val_dataset   = PadChestDataset(df_val_final,   IMAGE_DIR, mlb,
                                transform=val_transform,   return_meta=True)
test_dataset  = PadChestDataset(df_test_final,  IMAGE_DIR, mlb,
                                transform=val_transform,   return_meta=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=64,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=64,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# ═══════════════════════════════════════════════════════════════════════════════
# 7. VÉRIFICATION
# ═══════════════════════════════════════════════════════════════════════════════
imgs, metas, labels = next(iter(train_loader))
print(f"\n✅ DataLoader OK")
print(f"   Images : {imgs.shape}   → (B, 3, 224, 224)")
print(f"   Metas  : {metas.shape}  → (B, 5)")
print(f"   Labels : {labels.shape} → (B, {len(TARGET_LABELS)})")
print(f"   Pixels — min: {imgs.min():.3f} | max: {imgs.max():.3f} | mean: {imgs.mean():.3f}")
active = [TARGET_LABELS[i] for i, v in enumerate(labels[0]) if v == 1]
print(f"   Labels batch[0] : {active}")

## 2.Data augmentation (Flip, horizontal, rotation ±10° Zoom, brightness, contrast GridDistortion (léger)):


In [ ]:
import albumentations as A

geo_transforms = [

    # ✅ Flip horizontal — plausible
    # Justification : les radios peuvent être acquises miroir selon
    # l'orientation du patient ou du détecteur. Le contenu pulmonaire
    # est symétrique à gauche/droite (sauf dextrocardie rare).
    A.HorizontalFlip(p=0.5),

    # ✅ Rotation légère — plausible
    # Justification : le patient n'est jamais parfaitement vertical,
    # légère inclinaison du tronc fréquente en pratique clinique.
    # Limite absolue : ±10°. Au-delà, le médiastin se déplace
    # artificiellement et fausse les labels cardiomegaly.
    A.Rotate(limit=10, border_mode=cv2.BORDER_CONSTANT,
             value=0, p=0.5),

    # ✅ Légère translation — plausible
    # Justification : centrage du patient pas toujours parfait,
    # surtout en vue AP au lit.
    A.ShiftScaleRotate(
        shift_limit=0.05,    # max 5% de décalage
        scale_limit=0.05,    # max 5% de zoom
        rotate_limit=8,      # redondant avec Rotate mais combinable
        border_mode=cv2.BORDER_CONSTANT,
        value=0,
        p=0.4
    ),
]

In [ ]:
intensity_transforms = [

    # ✅ Brightness/Contrast — plausible
    # Justification : variation d'exposition entre machines et
    # techniciens (kVp, mAs différents selon protocoles hospitaliers).
    A.RandomBrightnessContrast(
        brightness_limit=0.15,   # ±15% max — au-delà, irréaliste
        contrast_limit=0.15,
        p=0.5
    ),

    # ✅ Gamma correction — plausible
    # Justification : variation de la courbe de réponse des
    # détecteurs numériques (DR vs CR).
    A.RandomGamma(gamma_limit=(80, 120), p=0.3),

    # ✅ CLAHE — plausible et recommandé
    # Justification : améliore la visibilité locale des nodules
    # et patterns de ground glass. Simule le post-traitement
    # appliqué différemment selon les constructeurs (Siemens vs GE).
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.5),

    # ✅ Sharpen léger — plausible
    # Justification : algorithmes de netteté varient selon
    # les paramètres de reconstruction DICOM.
    A.Sharpen(alpha=(0.1, 0.3), lightness=(0.8, 1.2), p=0.2),
]

In [ ]:
noise_transforms = [

    # ✅ Gaussian Noise — plausible
    # Justification : bruit quantique du détecteur, variable selon
    # la dose de radiation utilisée.
    A.GaussNoise(var_limit=(5.0, 20.0), p=0.3),

    # ✅ Blur très léger — plausible
    # Justification : légère dégradation due au mouvement
    # respiratoire pendant l'acquisition.
    A.GaussianBlur(blur_limit=(3, 3), p=0.2),
]

In [ ]:
deform_transforms = [

    # ✅ Grid Distortion très légère — limite plausible
    # Justification : légère distorsion du détecteur plan,
    # variation de la géométrie selon le positionnement.
    # ⚠️ distort_limit DOIT rester ≤ 0.05 pour rester réaliste.
    A.GridDistortion(
        num_steps=5,
        distort_limit=0.05,
        border_mode=cv2.BORDER_CONSTANT,
        value=0,
        p=0.2
    ),
]

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

MEAN = [0.503, 0.503, 0.503]
STD  = [0.246, 0.246, 0.246]

# ═══════════════════════════════════════════════════════════════════════════════
# 1. TROUVER LE BON CHEMIN IMAGE
# ═══════════════════════════════════════════════════════════════════════════════
sample_row = df_train_final.iloc[0]
print(f"ImageDir : {sample_row['ImageDir']} (type: {type(sample_row['ImageDir'])})")
print(f"ImageID  : {sample_row['ImageID']}")

# Chercher automatiquement l'image dans tout le dataset
base = Path("/kaggle/input/datasets/wassimfouratelamri/padchest224-complete")
matches = list(base.rglob(str(sample_row['ImageID'])))
if matches:
    sample_path = matches[0]
    IMAGE_DIR   = sample_path.parent.parent  # remonter au dossier racine
    print(f"\n✅ Image trouvée : {sample_path}")
    print(f"✅ IMAGE_DIR corrigé : {IMAGE_DIR}")
else:
    # Chercher n'importe quel PNG pour tester
    any_png = list(base.rglob("*.png"))
    if any_png:
        sample_path = any_png[0]
        IMAGE_DIR   = sample_path.parent.parent
        print(f"\n⚠️  Image originale non trouvée")
        print(f"✅ PNG de test trouvé : {sample_path}")
        print(f"✅ IMAGE_DIR corrigé : {IMAGE_DIR}")
    else:
        raise FileNotFoundError("Aucun PNG trouvé dans le dataset !")

# ═══════════════════════════════════════════════════════════════════════════════
# 2. TRANSFORMS — API corrigée pour albumentations >= 1.4
# ═══════════════════════════════════════════════════════════════════════════════
train_transform = A.Compose([
    A.Resize(224, 224),

    # Géométriques — API v2 : border_mode accepte int directement
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=10, p=0.5),                    # ← supprimé border_mode/value
    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.05,
        rotate_limit=0,
        p=0.3
    ),                                              # ← supprimé border_mode/value

    # Intensité
    A.RandomBrightnessContrast(
        brightness_limit=0.15,
        contrast_limit=0.15,
        p=0.5
    ),
    A.RandomGamma(gamma_limit=(80, 120), p=0.3),
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.5),
    A.Sharpen(alpha=(0.1, 0.3), lightness=(0.8, 1.2), p=0.2),

    # Bruit — API v2 : GaussNoise utilise std_range au lieu de var_limit
    A.GaussNoise(std_range=(0.01, 0.05), p=0.3),  # ← var_limit → std_range
    A.GaussianBlur(blur_limit=(3, 3), p=0.2),

    # Déformation — API v2 : plus de border_mode/value
    A.GridDistortion(num_steps=5, distort_limit=0.05, p=0.2),

    # Normalisation — toujours en dernier
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(224, 224),
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])

# ═══════════════════════════════════════════════════════════════════════════════
# 3. VISUALISATION CORRIGÉE
# ═══════════════════════════════════════════════════════════════════════════════
def visualize_augmentations(image_path, transform, n=6):
    # Charger avec vérification
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)

    if img is None:
        raise FileNotFoundError(f"Impossible de lire : {image_path}")

    # Resize si nécessaire
    img = cv2.resize(img, (224, 224))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

    # Extraire uniquement les augmentations visuelles (sans Normalize/ToTensor)
    aug_transforms = [
        t for t in transform.transforms
        if not isinstance(t, (A.Normalize, ToTensorV2))
    ]
    aug_only = A.Compose(aug_transforms)

    fig, axes = plt.subplots(2, 3, figsize=(13, 9))
    axes = axes.flatten()

    # Image originale (sans augmentation)
    axes[0].imshow(img, cmap='gray', vmin=0, vmax=255)
    axes[0].set_title('Original', fontsize=11, fontweight='bold')
    axes[0].axis('off')

    # 5 versions augmentées
    aug_names = ['Géom. + Intensité', 'Bruit + CLAHE',
                 'Rotation + Contraste', 'Noise + Blur', 'Distortion']
    for i in range(1, n):
        augmented = aug_only(image=img_rgb)['image']
        # Reconvertir en gris pour l'affichage
        aug_gray = augmented[:, :, 0]
        axes[i].imshow(aug_gray, cmap='gray', vmin=0, vmax=255)
        axes[i].set_title(f'Aug {i} — {aug_names[i-1]}', fontsize=10)
        axes[i].axis('off')

    plt.suptitle(
        f'Vérification visuelle — Augmentations PadChest\n{Path(image_path).name}',
        fontsize=12, y=1.01
    )
    plt.tight_layout()
    out = '/kaggle/working/augmentations_check.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅ Sauvegardé → {out}")

# ═══════════════════════════════════════════════════════════════════════════════
# 4. LANCER LA VISUALISATION
# ═══════════════════════════════════════════════════════════════════════════════
visualize_augmentations(sample_path, train_transform, n=6)

# ═══════════════════════════════════════════════════════════════════════════════
# 5. METTRE À JOUR LE DATASET AVEC LE BON IMAGE_DIR
# ═══════════════════════════════════════════════════════════════════════════════
# Recréer les datasets avec le chemin corrigé
train_dataset = PadChestDataset(df_train_final, IMAGE_DIR, mlb,
                                transform=train_transform, return_meta=True)
val_dataset   = PadChestDataset(df_val_final,   IMAGE_DIR, mlb,
                                transform=val_transform,   return_meta=True)
test_dataset  = PadChestDataset(df_test_final,  IMAGE_DIR, mlb,
                                transform=val_transform,   return_meta=True)

train_loader = DataLoader(train_dataset, batch_size=32,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=64,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=64,
                          shuffle=False, num_workers=2, pin_memory=True)

# Vérification finale
imgs, metas, labels = next(iter(train_loader))
print(f"\n✅ DataLoader OK")
print(f"   Images : {imgs.shape}   → (B, 3, 224, 224)")
print(f"   Metas  : {metas.shape}  → (B, 5)")
print(f"   Labels : {labels.shape} → (B, {len(TARGET_LABELS)})")
print(f"   Pixels — min: {imgs.min():.3f} | max: {imgs.max():.3f} | mean: {imgs.mean():.3f}")

## 3. DataLoader 224×224, batch 32-64 num_workers=4, pin_memory cache disque si RAM ok:

### 1 — Prefetching GPU avec un DataLoader custom:

In [ ]:
import torch
import threading
from torch.utils.data import DataLoader

class CUDAPrefetcher:
    """
    Précharge le prochain batch sur GPU pendant que le batch
    actuel est traité. Élimine le temps d'attente CPU→GPU.
    Gain typique : +15 à +25% de vitesse d'entraînement.
    """
    def __init__(self, loader, device):
        self.loader = loader
        self.device = device
        self.stream = torch.cuda.Stream()
        self.iter   = None

    def __iter__(self):
        self.iter = iter(self.loader)
        self._preload()
        return self

    def _preload(self):
        try:
            self._next = next(self.iter)
        except StopIteration:
            self._next = None
            return

        with torch.cuda.stream(self.stream):
            if isinstance(self._next, (list, tuple)):
                self._next = [
                    x.to(self.device, non_blocking=True)
                    if isinstance(x, torch.Tensor) else x
                    for x in self._next
                ]

    def __next__(self):
        torch.cuda.current_stream().wait_stream(self.stream)
        batch = self._next
        if batch is None:
            raise StopIteration
        self._preload()
        return batch

    def __len__(self):
        return len(self.loader)


# Utilisation
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_prefetcher = CUDAPrefetcher(train_loader, DEVICE)
val_prefetcher   = CUDAPrefetcher(val_loader,   DEVICE)

### 2 — Cache en mémoire RAM pour Kaggle:

In [ ]:
import numpy as np
from torch.utils.data import Dataset
from pathlib import Path
import cv2, torch, pandas as pd
from tqdm import tqdm

class PadChestDatasetCached(Dataset):
    """
    Version avec cache RAM du PadChestDataset.
    - Premier epoch  : lit depuis disque + met en cache (lent)
    - Epochs suivants : lit depuis RAM (très rapide)

    ⚠️  Kaggle RAM disponible : ~13 GB
        224×224×3 uint8 = ~150 KB/image
        Cache max recommandé : ~80 000 images (~12 GB)
    """
    def __init__(self, df, image_dir, mlb,
                 transform=None, return_meta=True,
                 use_cache=True, max_cache=80_000):

        self.df          = df.reset_index(drop=True)
        self.image_dir   = Path(image_dir)
        self.mlb         = mlb
        self.transform   = transform
        self.return_meta = return_meta
        self.use_cache   = use_cache
        self.cache       = {}

        self.y = mlb.transform(self.df['Labels'].tolist())

        if return_meta:
            self._build_meta()

        # Pré-charger en RAM si activé
        if use_cache:
            n = min(len(self.df), max_cache)
            print(f"Chargement cache RAM ({n:,} images)...")
            for idx in tqdm(range(n)):
                self.cache[idx] = self._load_image(idx)
            print(f"✅ Cache OK — {len(self.cache):,} images en RAM")

    def _build_meta(self):
        df   = self.df
        age  = pd.to_numeric(df['Age'],  errors='coerce').fillna(61)
        sex  = df['PatientSex_DICOM'].fillna('M')
        proj = df['Projection'].fillna('PA')
        self.meta = np.column_stack([
            ((age - 18) / 82).clip(0, 1).values,
            (sex  == 'M').astype(float).values,
            (sex  == 'F').astype(float).values,
            (proj == 'PA').astype(float).values,
            (proj == 'AP').astype(float).values,
        ]).astype(np.float32)

    def _load_image(self, idx):
        """Charge une image depuis disque → numpy uint8 RGB."""
        row      = self.df.iloc[idx]
        subdir   = str(int(float(str(row['ImageDir']))))
        filename = str(row['ImageID'])
        path     = self.image_dir / subdir / filename

        img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            img = np.zeros((224, 224), dtype=np.uint8)
        else:
            img = cv2.resize(img, (224, 224))

        return cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)  # (224, 224, 3) uint8

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Lire depuis cache ou disque
        if idx in self.cache:
            img = self.cache[idx].copy()  # copy() pour éviter les mutations
        else:
            img = self._load_image(idx)

        # Augmentations
        if self.transform:
            img = self.transform(image=img)['image']

        label = torch.tensor(self.y[idx], dtype=torch.float32)

        if self.return_meta:
            meta = torch.tensor(self.meta[idx], dtype=torch.float32)
            return img, meta, label

        return img, label

### 3 — DataLoaders optimisés:

In [ ]:
import psutil, os

def get_optimal_workers():
    """
    Calcule le nombre optimal de workers selon l'environnement.
    Kaggle : 4 CPUs disponibles → 2 workers recommandés
    (laisser 2 CPUs libres pour le process principal + GPU transfers)
    """
    cpu_count = psutil.cpu_count(logical=False) or 2
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        return min(2, cpu_count)          # Kaggle : limiter à 2
    return min(4, cpu_count)              # Local  : jusqu'à 4


NUM_WORKERS = get_optimal_workers()
print(f"Workers sélectionnés : {NUM_WORKERS}")

# ── RAM disponible → décider du cache ────────────────────────────────────────
ram_gb = psutil.virtual_memory().available / 1e9
use_cache = ram_gb > 8.0
print(f"RAM disponible : {ram_gb:.1f} GB → cache {'activé' if use_cache else 'désactivé'}")

# ── Instanciation avec cache ──────────────────────────────────────────────────
train_dataset_cached = PadChestDatasetCached(
    df_train_final, IMAGE_DIR, mlb,
    transform=train_transform,
    return_meta=True,
    use_cache=use_cache,
    max_cache=80_000
)
val_dataset_cached = PadChestDatasetCached(
    df_val_final, IMAGE_DIR, mlb,
    transform=val_transform,
    return_meta=True,
    use_cache=use_cache,
    max_cache=25_000
)

# ── DataLoaders avec tous les paramètres d'optimisation ──────────────────────
train_loader = DataLoader(
    train_dataset_cached,
    batch_size  = 32,
    shuffle     = True,
    num_workers = NUM_WORKERS,
    pin_memory  = True,          # copie en mémoire paginée → transfer GPU plus rapide
    drop_last   = True,          # évite un batch incomplet en fin d'epoch
    persistent_workers = True,   # garde les workers vivants entre epochs
    prefetch_factor    = 2,      # chaque worker pré-charge 2 batches en avance
)
val_loader = DataLoader(
    val_dataset_cached,
    batch_size  = 64,            # batch plus grand en val (pas de gradient)
    shuffle     = False,
    num_workers = NUM_WORKERS,
    pin_memory  = True,
    persistent_workers = True,
    prefetch_factor    = 2,
)
test_loader = DataLoader(
    val_dataset_cached,          # même transform que val
    batch_size  = 64,
    shuffle     = False,
    num_workers = NUM_WORKERS,
    pin_memory  = True,
)

# ── Activer le prefetcher GPU ─────────────────────────────────────────────────
if torch.cuda.is_available():
    train_prefetcher = CUDAPrefetcher(train_loader, DEVICE)
    val_prefetcher   = CUDAPrefetcher(val_loader,   DEVICE)
    print("✅ CUDA Prefetcher activé")
else:
    train_prefetcher = train_loader
    val_prefetcher   = val_loader
    print("⚠️  CPU seulement — prefetcher désactivé")


### 4 — Benchmark de vitesse:

In [ ]:
import time

def benchmark_loader(loader, name, n_batches=20):
    """Mesure le débit réel du DataLoader en images/seconde."""
    times = []
    loader_iter = iter(loader)

    # Warmup
    for _ in range(3):
        _ = next(loader_iter)

    for i in range(n_batches):
        t0 = time.perf_counter()
        batch = next(loader_iter)
        if isinstance(batch, (list, tuple)):
            imgs = batch[0]
        else:
            imgs = batch
        # Forcer le transfer GPU
        if torch.cuda.is_available():
            imgs = imgs.cuda(non_blocking=True)
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)

    imgs_per_sec = 32 / np.mean(times)
    print(f"{name:<30} {imgs_per_sec:>8.0f} img/s  "
          f"(batch: {np.mean(times)*1000:.1f} ms ± {np.std(times)*1000:.1f})")

print("\n=== BENCHMARK DATALOADERS ===")
benchmark_loader(train_loader, "Train (sans prefetch)")
if torch.cuda.is_available():
    benchmark_loader(train_prefetcher, "Train (avec prefetch GPU)")
benchmark_loader(val_loader,   "Val")

### 5 — Vérification finale:

In [ ]:
# Test avec le prefetcher
batch = next(iter(train_prefetcher))
imgs, metas, labels = batch

print(f"\n✅ Pipeline complet opérationnel")
print(f"   Device  : {DEVICE}")
print(f"   Images  : {imgs.shape}  sur {imgs.device}")
print(f"   Metas   : {metas.shape} sur {metas.device}")
print(f"   Labels  : {labels.shape}")
print(f"   Pixels  : min={imgs.min():.3f} | max={imgs.max():.3f} | mean={imgs.mean():.3f}")
print(f"\n   Train batches/epoch : {len(train_loader):,}")
print(f"   Val   batches/epoch : {len(val_loader):,}")

# Phase 4 — Modélisation (architectures CNN):

## 1. Baseline ResNet50 (Transfer learning ImageNet, Tête FC multi-label, BCEWithLogitsLoss):


In [ ]:
N_CLASSES = 19
N_META = 5

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import timm
import numpy as np
from torch.amp import GradScaler, autocast   # ← API moderne (plus de cuda.amp)
from sklearn.metrics import roc_auc_score, f1_score, average_precision_score
import time, os, json, matplotlib.pyplot as plt

# ═══════════════════════════════════════════════════════════════════════════════
# 1. CONFIG
# ═══════════════════════════════════════════════════════════════════════════════
CONFIG = {
    'backbone'       : 'resnet50',
    'pretrained'     : True,
    'n_classes'      : N_CLASSES,
    'n_meta_features': N_META,
    'epochs'         : 30,
    'batch_size'     : 32,
    'lr_head'        : 1e-3,
    'lr_backbone'    : 1e-4,
    'weight_decay'   : 1e-4,
    'patience'       : 7,
    'unfreeze_epoch' : 5,
    'checkpoint_dir' : '/kaggle/working/checkpoints',
    'log_path'       : '/kaggle/working/training_log.json',
    'device'         : 'cuda' if torch.cuda.is_available() else 'cpu',
    'amp'            : True,
}
os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)
DEVICE     = torch.device(CONFIG['device'])
AMP_DEVICE = CONFIG['device']              # ← string 'cuda' requis par nouvelle API
USE_AMP    = CONFIG['amp'] and CONFIG['device'] == 'cuda'
print(f"Device : {DEVICE}  |  AMP : {USE_AMP}")

# ═══════════════════════════════════════════════════════════════════════════════
# 2. MODÈLE
# ═══════════════════════════════════════════════════════════════════════════════
class ResNet50PadChest(nn.Module):
    def __init__(self, n_classes=N_CLASSES, n_meta=N_META, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            'resnet50', pretrained=pretrained,
            num_classes=0, global_pool='avg'
        )
        backbone_dim = self.backbone.num_features   # 2048
        self.meta_branch = nn.Sequential(
            nn.Linear(n_meta, 64), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(64, 32), nn.ReLU()
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(backbone_dim + 32, 512), nn.BatchNorm1d(512), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(512, n_classes)
        )
        for m in [self.meta_branch, self.classifier]:
            for layer in m.modules():
                if isinstance(layer, nn.Linear):
                    nn.init.kaiming_normal_(layer.weight, mode='fan_out')
                    nn.init.zeros_(layer.bias)

    def forward(self, image, metadata):
        return self.classifier(
            torch.cat([self.backbone(image), self.meta_branch(metadata)], dim=1)
        )

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False
        print("🔒 Backbone gelé")

    def unfreeze_backbone(self, lr=1e-4):
        for p in self.backbone.parameters(): p.requires_grad = True
        print(f"🔓 Backbone dégelé (lr={lr})")

    def get_param_groups(self, lr_head, lr_backbone):
        return [
            {'params': list(self.backbone.parameters()),                        'lr': lr_backbone},
            {'params': list(self.meta_branch.parameters()) +
                       list(self.classifier.parameters()),                      'lr': lr_head},
        ]

model = ResNet50PadChest(
    n_classes=CONFIG['n_classes'],
    n_meta=CONFIG['n_meta_features'],
    pretrained=CONFIG['pretrained']
).to(DEVICE)
model.freeze_backbone()

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Params totaux : {total:,}  |  Trainables : {trainable:,} ({100*trainable/total:.1f}%)")

# ═══════════════════════════════════════════════════════════════════════════════
# 3. LOSS
# ═══════════════════════════════════════════════════════════════════════════════
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, targets):
        bce   = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p_t   = torch.sigmoid(logits) * targets + (1 - torch.sigmoid(logits)) * (1 - targets)
        loss  = ((1 - p_t) ** self.gamma) * bce
        if self.alpha is not None:
            loss = loss * self.alpha.to(logits.device)
        return loss.mean()

y_train_np  = mlb.transform(df_train_final['Labels'].tolist())
counts      = y_train_np.sum(axis=0)
pos_weights = torch.tensor(
    np.clip((len(y_train_np) - counts) / (counts + 1e-6), 1.0, 20.0),
    dtype=torch.float32
)
criterion = FocalLoss(gamma=2.0, alpha=pos_weights).to(DEVICE)
print(f"Class weights — min: {pos_weights.min():.2f} | max: {pos_weights.max():.2f}")

# ═══════════════════════════════════════════════════════════════════════════════
# 4. OPTIMISEUR + SCALER (API corrigée)
# ═══════════════════════════════════════════════════════════════════════════════
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CONFIG['lr_head'], weight_decay=CONFIG['weight_decay']
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CONFIG['epochs'], eta_min=1e-6
)

# ── Fix FutureWarning : torch.amp.GradScaler('cuda') au lieu de cuda.amp ──────
scaler = GradScaler(AMP_DEVICE, enabled=USE_AMP)

# ═══════════════════════════════════════════════════════════════════════════════
# 5. MÉTRIQUES
# ═══════════════════════════════════════════════════════════════════════════════
def compute_metrics(y_true, y_probs, threshold=0.5):
    aucs, aps = [], []
    for i in range(y_true.shape[1]):
        if y_true[:, i].sum() > 0:
            aucs.append(roc_auc_score(y_true[:, i], y_probs[:, i]))
            aps.append(average_precision_score(y_true[:, i], y_probs[:, i]))
    y_pred = (y_probs >= threshold).astype(int)
    return {
        'auc_macro': np.mean(aucs) if aucs else 0.0,
        'map'      : np.mean(aps)  if aps  else 0.0,
        'f1_macro' : f1_score(y_true, y_pred, average='macro',  zero_division=0),
        'f1_micro' : f1_score(y_true, y_pred, average='micro',  zero_division=0),
    }

# ═══════════════════════════════════════════════════════════════════════════════
# 6. TRAIN / EVAL — autocast avec device_type string
# ═══════════════════════════════════════════════════════════════════════════════
def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss = 0.0
    for imgs, metas, labels in loader:
        imgs, metas, labels = (
            imgs.to(DEVICE, non_blocking=True),
            metas.to(DEVICE, non_blocking=True),
            labels.to(DEVICE, non_blocking=True)
        )
        optimizer.zero_grad(set_to_none=True)

        # ── Fix FutureWarning : autocast(device_type=...) ─────────────────────
        with autocast(device_type=AMP_DEVICE, enabled=USE_AMP):
            loss = criterion(model(imgs, metas), labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, all_probs, all_labels = 0.0, [], []
    for imgs, metas, labels in loader:
        imgs, metas, labels = (
            imgs.to(DEVICE, non_blocking=True),
            metas.to(DEVICE, non_blocking=True),
            labels.to(DEVICE, non_blocking=True)
        )
        with autocast(device_type=AMP_DEVICE, enabled=USE_AMP):
            logits = model(imgs, metas)
            loss   = criterion(logits, labels)
        total_loss  += loss.item()
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    y_probs  = np.vstack(all_probs)
    y_true   = np.vstack(all_labels)
    metrics  = compute_metrics(y_true, y_probs)
    metrics['loss'] = total_loss / len(loader)
    return metrics, y_probs, y_true

# ═══════════════════════════════════════════════════════════════════════════════
# 7. TRAINING LOOP
# ═══════════════════════════════════════════════════════════════════════════════
best_auc, patience_count, history = 0.0, 0, []
backbone_unfrozen = False
ckpt_path = f"{CONFIG['checkpoint_dir']}/best_resnet50.pt"

print(f"\n{'='*65}")
print(f"Entraînement ResNet50 — {CONFIG['epochs']} epochs — {DEVICE}")
print(f"{'='*65}\n")

for epoch in range(1, CONFIG['epochs'] + 1):
    t0 = time.time()

    # Dégeler backbone
    if epoch == CONFIG['unfreeze_epoch'] and not backbone_unfrozen:
        model.unfreeze_backbone(CONFIG['lr_backbone'])
        backbone_unfrozen = True
        optimizer = optim.AdamW(
            model.get_param_groups(CONFIG['lr_head'], CONFIG['lr_backbone']),
            weight_decay=CONFIG['weight_decay']
        )
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=CONFIG['epochs'] - epoch, eta_min=1e-6
        )
        print(f"  → Optimiseur réinitialisé lr_head={CONFIG['lr_head']} | lr_backbone={CONFIG['lr_backbone']}")

    train_loss          = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
    val_metrics, _, _   = evaluate(model, val_loader, criterion)

    scheduler.step()

    lr  = optimizer.param_groups[-1]['lr']
    log = {
        'epoch': epoch, 'train_loss': round(train_loss, 4),
        'val_loss': round(val_metrics['loss'], 4),
        'val_auc' : round(val_metrics['auc_macro'], 4),
        'val_map' : round(val_metrics['map'], 4),
        'val_f1'  : round(val_metrics['f1_macro'], 4),
        'lr'      : round(lr, 7), 'time_s': round(time.time() - t0, 1)
    }
    history.append(log)

    flag = '✅' if val_metrics['auc_macro'] > best_auc else '  '
    print(f"Ep {epoch:02d}/{CONFIG['epochs']} {flag} "
          f"loss {train_loss:.4f}→{val_metrics['loss']:.4f} "
          f"AUC {val_metrics['auc_macro']:.4f} "
          f"mAP {val_metrics['map']:.4f} "
          f"F1 {val_metrics['f1_macro']:.4f} "
          f"lr {lr:.2e} {log['time_s']:.0f}s")

    if val_metrics['auc_macro'] > best_auc:
        best_auc, patience_count = val_metrics['auc_macro'], 0

        # ── Fix UnpicklingError : sauvegarder UNIQUEMENT model_state ──────────
        # Ne pas inclure CONFIG (dict avec strings) ni TARGET_LABELS (liste)
        # car PyTorch 2.6 weights_only=True refuse les types non-tensor
        torch.save(
            model.state_dict(),          # ← state_dict pur, pas de dict wrapper
            ckpt_path
        )
        # Sauvegarder les métadonnées séparément en JSON (pas de pickle)
        with open(ckpt_path.replace('.pt', '_meta.json'), 'w') as f:
            json.dump({
                'epoch': epoch, 'best_auc': best_auc,
                'val_metrics': {k: float(v) for k, v in val_metrics.items()},
                'target_labels': TARGET_LABELS, 'config': CONFIG
            }, f, indent=2)
        print(f"  ✅ Meilleur modèle sauvegardé (AUC={best_auc:.4f})")
    else:
        patience_count += 1
        if patience_count >= CONFIG['patience']:
            print(f"\n⏹️  Early stopping epoch {epoch}")
            break

with open(CONFIG['log_path'], 'w') as f:
    json.dump(history, f, indent=2)
print(f"\n✅ Historique sauvegardé")

# ═══════════════════════════════════════════════════════════════════════════════
# 8. CHARGEMENT + ÉVALUATION TEST — Fix UnpicklingError
# ═══════════════════════════════════════════════════════════════════════════════
# ── Fix : weights_only=True + charger state_dict pur ─────────────────────────
model.load_state_dict(
    torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
)

# Charger les métadonnées depuis JSON
with open(ckpt_path.replace('.pt', '_meta.json')) as f:
    ckpt_meta = json.load(f)
print(f"\nMeilleur checkpoint — epoch {ckpt_meta['epoch']} "
      f"| val AUC {ckpt_meta['best_auc']:.4f}")

test_metrics, y_probs_test, y_true_test = evaluate(model, test_loader, criterion)

print(f"\n{'='*50}")
print(f"RÉSULTATS FINAUX — TEST SET")
print(f"{'='*50}")
for k, v in test_metrics.items():
    print(f"  {k:<15} : {v:.4f}")

print(f"\nAUC par pathologie :")
for i, label in enumerate(TARGET_LABELS):
    if y_true_test[:, i].sum() > 0:
        auc = roc_auc_score(y_true_test[:, i], y_probs_test[:, i])
        print(f"  {label:<28} {auc:.4f}  {'█' * int(auc * 20)}")

# ═══════════════════════════════════════════════════════════════════════════════
# 9. COURBES D'ENTRAÎNEMENT
# ═══════════════════════════════════════════════════════════════════════════════
epochs_log = [h['epoch'] for h in history]
fig, axes  = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs_log, [h['train_loss'] for h in history], label='Train')
axes[0].plot(epochs_log, [h['val_loss']   for h in history], label='Val')
axes[0].axvline(CONFIG['unfreeze_epoch'], color='red', linestyle='--',
                alpha=0.5, label='Unfreeze')
axes[0].set_title('Loss') ; axes[0].legend() ; axes[0].grid(alpha=0.3)

axes[1].plot(epochs_log, [h['val_auc'] for h in history], color='green')
axes[1].axhline(best_auc, color='green', linestyle='--',
                alpha=0.5, label=f'Best {best_auc:.4f}')
axes[1].set_title('Val AUC Macro') ; axes[1].legend() ; axes[1].grid(alpha=0.3)

axes[2].plot(epochs_log, [h['val_f1'] for h in history], color='orange')
axes[2].set_title('Val F1 Macro') ; axes[2].grid(alpha=0.3)

plt.suptitle("Courbes d'entraînement — ResNet50 PadChest", fontsize=13)
plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Courbes sauvegardées → training_curves.png")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import numpy as np
import json, os, time
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_DEVICE = DEVICE.type
USE_AMP    = DEVICE.type == 'cuda'

# ═══════════════════════════════════════════════════════════════════════════════
# FIX 1 — Recréer les DataLoaders stables pour Kaggle
# ═══════════════════════════════════════════════════════════════════════════════
def make_loaders(train_ds, val_ds, test_ds, batch_size=32):
    """
    DataLoaders optimisés pour Kaggle :
    - num_workers=2 (pas 4, évite les crashes)
    - persistent_workers=False (cause principale du crash)
    - prefetch_factor=2 seulement si num_workers > 0
    - pin_memory=True uniquement sur CUDA
    """
    pin = DEVICE.type == 'cuda'

    train_loader = DataLoader(
        train_ds,
        batch_size  = batch_size,
        shuffle     = True,
        num_workers = 2,
        pin_memory  = pin,
        drop_last   = True,
        persistent_workers = False,   # ← FIX principal
        prefetch_factor    = 2,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size  = 64,
        shuffle     = False,
        num_workers = 2,
        pin_memory  = pin,
        persistent_workers = False,   # ← FIX principal
        prefetch_factor    = 2,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size  = 64,
        shuffle     = False,
        num_workers = 2,
        pin_memory  = pin,
        persistent_workers = False,
        prefetch_factor    = 2,
    )
    print(f"✅ DataLoaders recréés")
    print(f"   Train : {len(train_loader):,} batches | "
          f"Val : {len(val_loader):,} | Test : {len(test_loader):,}")
    return train_loader, val_loader, test_loader

# Recréer avec les datasets existants
train_loader, val_loader, test_loader = make_loaders(
    train_dataset, val_dataset, test_dataset, batch_size=32
)

# Test rapide pour vérifier que le loader fonctionne
try:
    imgs, metas, labels = next(iter(train_loader))
    print(f"✅ Loader OK — batch: {imgs.shape}, metas: {metas.shape}, labels: {labels.shape}")
except Exception as e:
    print(f"❌ Loader KO : {e}")
    print("→ Passage en num_workers=0 (mode debug)")
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=0)
    test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=0)


# ═══════════════════════════════════════════════════════════════════════════════
# ARCHITECTURES
# ═══════════════════════════════════════════════════════════════════════════════
class DenseNet121PadChest(nn.Module):
    def __init__(self, n_classes=N_CLASSES, n_meta=N_META, pretrained=True, dropout=0.4):
        super().__init__()
        self.backbone = timm.create_model(
            'densenet121', pretrained=pretrained, num_classes=0, global_pool='avg'
        )
        feat_dim = self.backbone.num_features
        self.meta_branch = nn.Sequential(
            nn.Linear(n_meta, 64), nn.BatchNorm1d(64), nn.ReLU(inplace=True),
            nn.Dropout(0.3), nn.Linear(64, 32), nn.ReLU(inplace=True)
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(feat_dim + 32, 512),
            nn.BatchNorm1d(512), nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.75), nn.Linear(512, n_classes)
        )
        for m in [self.meta_branch, self.classifier]:
            for layer in m.modules():
                if isinstance(layer, nn.Linear):
                    nn.init.kaiming_normal_(layer.weight)
                    nn.init.zeros_(layer.bias)

    def forward(self, image, metadata):
        return self.classifier(
            torch.cat([self.backbone(image), self.meta_branch(metadata)], dim=1)
        )
    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False
    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = True
    def get_param_groups(self, lr_head=1e-3, lr_backbone=1e-4):
        return [
            {'params': self.backbone.parameters(),    'lr': lr_backbone},
            {'params': self.meta_branch.parameters(), 'lr': lr_head},
            {'params': self.classifier.parameters(),  'lr': lr_head},
        ]


class EfficientNetB4PadChest(nn.Module):
    def __init__(self, n_classes=N_CLASSES, n_meta=N_META, pretrained=True, dropout=0.4):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b4', pretrained=pretrained,
            num_classes=0, global_pool='avg', drop_rate=0.3
        )
        feat_dim = self.backbone.num_features
        self.meta_branch = nn.Sequential(
            nn.Linear(n_meta, 64), nn.BatchNorm1d(64), nn.ReLU(inplace=True),
            nn.Dropout(0.3), nn.Linear(64, 32), nn.ReLU(inplace=True)
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(feat_dim + 32, 768),
            nn.BatchNorm1d(768), nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.6), nn.Linear(768, 256),
            nn.ReLU(inplace=True), nn.Dropout(dropout * 0.4),
            nn.Linear(256, n_classes)
        )
        for m in [self.meta_branch, self.classifier]:
            for layer in m.modules():
                if isinstance(layer, nn.Linear):
                    nn.init.kaiming_normal_(layer.weight)
                    nn.init.zeros_(layer.bias)

    def forward(self, image, metadata):
        return self.classifier(
            torch.cat([self.backbone(image), self.meta_branch(metadata)], dim=1)
        )
    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False
    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = True
    def get_param_groups(self, lr_head=1e-3, lr_backbone=1e-4):
        return [
            {'params': self.backbone.parameters(),    'lr': lr_backbone},
            {'params': self.meta_branch.parameters(), 'lr': lr_head},
            {'params': self.classifier.parameters(),  'lr': lr_head},
        ]


def build_model(name, n_classes=N_CLASSES, n_meta=N_META, pretrained=True):
    models_map = {
        'densenet121'    : DenseNet121PadChest,
        'efficientnet_b4': EfficientNetB4PadChest,
    }
    assert name in models_map
    model = models_map[name](n_classes, n_meta, pretrained).to(DEVICE)
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n[{name}]  {total/1e6:.1f}M params  |  backbone dim : {model.backbone.num_features}")
    return model


# ═══════════════════════════════════════════════════════════════════════════════
# TRAINER — avec FIX 2 : recréer le loader à chaque epoch si nécessaire
# ═══════════════════════════════════════════════════════════════════════════════
class PadChestTrainer:
    def __init__(self, model, model_name, train_loader, val_loader,
                 criterion, config):
        self.model        = model
        self.model_name   = model_name
        self.train_loader = train_loader
        self.val_loader   = val_loader
        self.criterion    = criterion
        self.config       = config
        self.history      = []
        self.best_auc     = 0.0
        self.patience_ctr = 0
        self.scaler       = GradScaler(AMP_DEVICE, enabled=USE_AMP)
        self.ckpt_path    = f"{config['checkpoint_dir']}/best_{model_name}.pt"
        self.meta_path    = self.ckpt_path.replace('.pt', '_meta.json')
        os.makedirs(config['checkpoint_dir'], exist_ok=True)
        self.model.freeze_backbone()
        self._build_optimizer(phase=1)

    def _build_optimizer(self, phase):
        if phase == 1:
            params = filter(lambda p: p.requires_grad, self.model.parameters())
            self.optimizer = torch.optim.AdamW(
                params, lr=self.config['lr_head'],
                weight_decay=self.config['weight_decay']
            )
        else:
            self.optimizer = torch.optim.AdamW(
                self.model.get_param_groups(
                    self.config['lr_head'], self.config['lr_backbone']
                ),
                weight_decay=self.config['weight_decay']
            )
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer, T_max=self.config['epochs'], eta_min=1e-6
        )

    def _safe_loader(self, loader):
        """
        FIX 2 : wrapper qui relance le loader avec num_workers=0
        si un worker crash est détecté.
        """
        try:
            for batch in loader:
                yield batch
        except RuntimeError as e:
            if 'worker' in str(e).lower():
                print(f"\n⚠️  Worker crash détecté → bascule num_workers=0")
                ds  = loader.dataset
                bs  = loader.batch_size
                fallback = DataLoader(ds, batch_size=bs, shuffle=True, num_workers=0)
                self.train_loader = fallback
                for batch in fallback:
                    yield batch
            else:
                raise

    def _train_epoch(self):
        self.model.train()
        total_loss = 0.0
        n = 0
        for imgs, metas, labels in self._safe_loader(self.train_loader):
            imgs   = imgs.to(DEVICE, non_blocking=True)
            metas  = metas.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            self.optimizer.zero_grad(set_to_none=True)
            with autocast(device_type=AMP_DEVICE, enabled=USE_AMP):
                logits = self.model(imgs, metas)
                loss   = self.criterion(logits, labels)
            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.optimizer)
            nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.scaler.step(self.optimizer)
            self.scaler.update()
            total_loss += loss.item()
            n += 1
        return total_loss / max(n, 1)

    @torch.no_grad()
    def _eval_epoch(self, loader):
        self.model.eval()
        all_probs, all_labels, total_loss = [], [], 0.0
        for imgs, metas, labels in loader:
            imgs   = imgs.to(DEVICE, non_blocking=True)
            metas  = metas.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            with autocast(device_type=AMP_DEVICE, enabled=USE_AMP):
                logits = self.model(imgs, metas)
                loss   = self.criterion(logits, labels)
            total_loss  += loss.item()
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(labels.cpu().numpy())

        y_probs = np.vstack(all_probs)
        y_true  = np.vstack(all_labels)
        aucs, aps = [], []
        for i in range(y_true.shape[1]):
            if y_true[:, i].sum() > 0:
                aucs.append(roc_auc_score(y_true[:, i], y_probs[:, i]))
                aps.append(average_precision_score(y_true[:, i], y_probs[:, i]))
        y_pred = (y_probs >= 0.5).astype(int)
        return {
            'loss'     : total_loss / len(loader),
            'auc_macro': np.mean(aucs) if aucs else 0.0,
            'map'      : np.mean(aps)  if aps  else 0.0,
            'f1_macro' : f1_score(y_true, y_pred, average='macro',  zero_division=0),
            'f1_micro' : f1_score(y_true, y_pred, average='micro',  zero_division=0),
        }, y_probs, y_true

    def train(self):
        cfg = self.config
        print(f"\n{'='*60}")
        print(f"Entraînement {self.model_name} — {DEVICE}")
        print(f"{'='*60}")
        backbone_unfrozen = False

        for epoch in range(1, cfg['epochs'] + 1):
            t0 = time.time()
            if epoch == cfg['unfreeze_epoch'] and not backbone_unfrozen:
                self.model.unfreeze_backbone()
                self._build_optimizer(phase=2)
                backbone_unfrozen = True
                print(f"  [Epoch {epoch}] 🔓 Backbone dégelé")

            train_loss          = self._train_epoch()
            val_metrics, yp, yt = self._eval_epoch(self.val_loader)
            self.scheduler.step()

            lr  = self.optimizer.param_groups[-1]['lr']
            log = {
                'epoch': epoch, 'train_loss': round(train_loss, 4),
                **{f'val_{k}': round(float(v), 4) for k, v in val_metrics.items()},
                'lr': round(lr, 7), 'time_s': round(time.time() - t0, 1)
            }
            self.history.append(log)

            improved = '✅' if val_metrics['auc_macro'] > self.best_auc else '  '
            print(f"Ep {epoch:02d}/{cfg['epochs']} {improved} "
                  f"loss {train_loss:.4f}→{val_metrics['loss']:.4f} "
                  f"AUC {val_metrics['auc_macro']:.4f} "
                  f"mAP {val_metrics['map']:.4f} "
                  f"F1 {val_metrics['f1_macro']:.4f} "
                  f"lr {lr:.1e} {log['time_s']:.0f}s")

            if val_metrics['auc_macro'] > self.best_auc:
                self.best_auc     = val_metrics['auc_macro']
                self.patience_ctr = 0
                torch.save(self.model.state_dict(), self.ckpt_path)
                with open(self.meta_path, 'w') as f:
                    json.dump({
                        'epoch'      : epoch,
                        'best_auc'   : float(self.best_auc),
                        'val_metrics': {k: float(v) for k, v in val_metrics.items()}
                    }, f, indent=2)
                print(f"  ✅ Checkpoint sauvegardé (AUC={self.best_auc:.4f})")
            else:
                self.patience_ctr += 1
                if self.patience_ctr >= cfg['patience']:
                    print(f"\n⏹️  Early stopping epoch {epoch}")
                    break

        with open(f"{cfg['checkpoint_dir']}/{self.model_name}_history.json", 'w') as f:
            json.dump(self.history, f, indent=2)
        print(f"\nBest val AUC : {self.best_auc:.4f}")
        return self.best_auc

    def evaluate_test(self, test_loader):
        self.model.load_state_dict(
            torch.load(self.ckpt_path, map_location=DEVICE, weights_only=True)
        )
        with open(self.meta_path) as f:
            meta = json.load(f)
        print(f"\nCheckpoint chargé — epoch {meta['epoch']} "
              f"| val AUC {meta['best_auc']:.4f}")
        test_metrics, y_probs, y_true = self._eval_epoch(test_loader)
        print(f"\n{'─'*45}\nTEST SET — {self.model_name}\n{'─'*45}")
        for k, v in test_metrics.items():
            print(f"  {k:<15} : {v:.4f}")
        print(f"\nAUC par pathologie :")
        for i, label in enumerate(TARGET_LABELS):
            if y_true[:, i].sum() > 0:
                auc = roc_auc_score(y_true[:, i], y_probs[:, i])
                print(f"  {label:<28} {auc:.4f}  {'█' * int(auc * 20)}")
        return test_metrics, y_probs, y_true


# ═══════════════════════════════════════════════════════════════════════════════
# LOSS + LANCEMENT
# ═══════════════════════════════════════════════════════════════════════════════
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
    def forward(self, logits, targets):
        bce  = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p_t  = torch.sigmoid(logits) * targets + (1 - torch.sigmoid(logits)) * (1 - targets)
        loss = ((1 - p_t) ** self.gamma) * bce
        if self.alpha is not None:
            loss = loss * self.alpha.to(logits.device)
        return loss.mean()

y_train_np  = mlb.transform(df_train_final['Labels'].tolist())
counts      = y_train_np.sum(axis=0)
pos_weights = torch.tensor(
    np.clip((len(y_train_np) - counts) / (counts + 1e-6), 1.0, 20.0),
    dtype=torch.float32
)
criterion = FocalLoss(gamma=2.0, alpha=pos_weights).to(DEVICE)

# ── FIG 7 : pos_weights par classe ──────────────────────────────────────────
weights_np = pos_weights.cpu().numpy()
sort_idx_w = np.argsort(weights_np)[::-1]

fig, ax = plt.subplots(figsize=(14, 5))
bar_colors_w = [
    '#E53935' if w > 10 else '#FB8C00' if w > 5 else '#43A047'
    for w in weights_np[sort_idx_w]
]
bars = ax.bar([TARGET_LABELS[i] for i in sort_idx_w], weights_np[sort_idx_w],
              color=bar_colors_w, edgecolor='white', linewidth=0.6)
for bar, w in zip(bars, weights_np[sort_idx_w]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
            f'{w:.1f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

legend_handles_w = [

    plt.Line2D([0], [0], color='#E53935', lw=8, label='Poids > 10 (très rare)'),test_metrics, y_probs_test, y_true_test = trainer.evaluate_test(test_loader)

    plt.Line2D([0], [0], color='#FB8C00', lw=8, label='Poids 5–10 (rare)'),best_auc = trainer.train()

    plt.Line2D([0], [0], color='#43A047', lw=8, label='Poids ≤ 5 (équilibré)'),)

]    model, 'densenet121', train_loader, val_loader, criterion, TRAIN_CONFIG

ax.legend(handles=legend_handles_w, fontsize=9)trainer = PadChestTrainer(

ax.set_title('Poids de classe pour la FocalLoss (pos_weights)', fontsize=14, fontweight='bold', pad=15)model   = build_model('densenet121', n_classes=len(TARGET_LABELS), n_meta=N_META)

ax.set_ylabel('Poids', fontsize=11)

ax.axhline(1.0, color='gray', linestyle=':', linewidth=1)}

ax.spines['top'].set_visible(False)    'checkpoint_dir': '/kaggle/working/checkpoints',

ax.spines['right'].set_visible(False)    'unfreeze_epoch': 5,

plt.xticks(rotation=40, ha='right', fontsize=9)    'patience'      : 7,

savefig('fig07_pos_weights.png')    'weight_decay'  : 1e-4,

    'lr_backbone'   : 1e-4,

TRAIN_CONFIG = {    'lr_head'       : 1e-3,
    'epochs'        : 30,

In [ ]:
# ── FIG 8 : Training curves ──────────────────────────────────────────────────

history = trainer.historyprint('\n✨ Prêt pour soumission Kaggle !')

epochs_log = [h['epoch'] for h in history]print(f'🖼️  Images : {len(df):,}')

print(f'🏥 Patients : {len(df_train_final) + len(df_val_final) + len(df_test_final):,}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))print(f'📋 Classes : {len(TARGET_LABELS)} pathologies')

axes[0].plot(epochs_log, [h['train_loss'] for h in history], label='Train', color='#1976D2')print(f'\n📈 Meilleure performance : AUC Macro = {best_auc:.4f}')

axes[0].plot(epochs_log, [h['val_loss'] for h in history], label='Val', color='#D32F2F')

axes[0].axvline(TRAIN_CONFIG['unfreeze_epoch'], color='red', linestyle='--', alpha=0.7, label='Unfreeze backbone')    print(f'  ✅ {fig}' if os.path.exists(path) else f'  ❌ {fig} (manquant)')

axes[0].set_title('Loss', fontweight='bold')    path = f'/kaggle/working/figures/{fig}'

axes[0].set_xlabel('Epoch')for fig in figures:

axes[0].legend()]

axes[0].grid(alpha=0.3)    'fig09_test_metrics_per_class.png'

    'fig08_training_curves.png',

axes[1].plot(epochs_log, [h['val_auc'] for h in history], color='#388E3C', linewidth=2)    'fig07_pos_weights.png',

axes[1].axhline(best_auc, color='#388E3C', linestyle='--', alpha=0.7, label=f'Best {best_auc:.4f}')    'fig06_metadata_features.png',

axes[1].set_title('Validation AUC Macro', fontweight='bold')    'fig05_class_prevalence.png',

axes[1].set_xlabel('Epoch')    'fig04_label_cooccurrence.png',

axes[1].legend()    'fig03_label_distribution.png',

axes[1].grid(alpha=0.3)    'fig02_patient_splits.png',

    'fig01_dataset_overview.png',

axes[2].plot(epochs_log, [h['val_f1'] for h in history], color='#F57C00', linewidth=2)figures = [

axes[2].set_title('Validation F1 Macro', fontweight='bold')print('\n📊 Figures sauvegardées dans /kaggle/working/figures/ :')

axes[2].set_xlabel('Epoch')print('🎉 Notebook finalisé avec succès !')

axes[2].grid(alpha=0.3)

import os

plt.suptitle('Courbes d\'entraînement — DenseNet121 + Metadata Fusion', fontsize=14, fontweight='bold', y=1.02)# ── STEP 10 : Final summary — All saved figures ──────────────────────────────

plt.tight_layout()

savefig('fig08_training_curves.png')print('✅ Fig 9 sauvegardée → fig09_test_metrics_per_class.png')

print('✅ Fig 8 sauvegardée → fig08_training_curves.png')savefig('fig09_test_metrics_per_class.png')

plt.tight_layout()

# ── FIG 9 : Test set evaluation metrics ──────────────────────────────────────plt.suptitle('Métriques d\'évaluation par classe — Test Set', fontsize=14, fontweight='bold', y=1.02)

labels = TARGET_LABELS

aucs = [test_metrics.get(f'auc_{i}', 0) for i in range(len(labels))]    ax.grid(axis='y', alpha=0.3)

aps  = [test_metrics.get(f'ap_{i}', 0) for i in range(len(labels))]    ax.tick_params(axis='x', rotation=45, labelsize=9)

f1s  = [test_metrics.get(f'f1_{i}', 0) for i in range(len(labels))]    ax.set_ylim(0, 1.05)

sort_idx = np.argsort(aucs)[::-1]    ax.set_title(title, fontweight='bold')

                f'{value:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,

for ax, values, title, color in zip(    for bar, value in zip(bars, values):

    axes,    bars = ax.bar([labels[i] for i in sort_idx], values, color=color, alpha=0.8, edgecolor='white', linewidth=0.5)

    [[aucs[i] for i in sort_idx], [aps[i] for i in sort_idx], [f1s[i] for i in sort_idx]],):

    ['AUC par classe (Test)', 'Average Precision par classe (Test)', 'F1 par classe (Test)'],    ['#4CAF50', '#2196F3', '#FF9800']